## SAMD9L · cromatina en ventana fija
Ejecución completa de 54 configuraciones y todas las alternativas. El diseño está incorporado. Los checkpoints se guardan en Google Drive para reanudar la ejecución si se interrumpe. Se solicita la clave en un campo oculto.

In [ ]:
# Instalar las dependencias en el mismo intérprete que ejecuta Colab.
import subprocess
import sys
from importlib.metadata import version

dependencies = ['alphagenome==0.9.0', 'pandas==2.2.3', 'pyarrow', 'scipy', 'statsmodels', 'matplotlib', 'seaborn']
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *dependencies], check=True)
if version("pandas") != "2.2.3" or version("alphagenome") != "0.9.0":
    raise RuntimeError("Revisa el resultado de la instalación antes de continuar.")
if "pandas" in sys.modules and sys.modules["pandas"].__version__ != "2.2.3":
    raise RuntimeError("pandas 2.2.3 ya está instalado. Reinicia la sesión de Colab y vuelve a ejecutar el cuaderno.")
import pandas as pd
print("pandas", pd.__version__, "| alphagenome", version("alphagenome"))


In [ ]:
# 2) Imports y configuración editable
import gc
import hashlib
import json
import math
import os
import re
import shutil
import time
import warnings
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from alphagenome.data import genome
from alphagenome.models import dna_client

# ---- Opciones de ejecución ----
MAX_PAIRS = None          # None = ambos; usa 1 para una prueba barata.
FORCE_RERUN = False       # True vuelve a solicitar pares ya guardados.
API_RETRIES = 3
RETRY_BASE_SECONDS = 5

# ---- Definiciones del análisis ----
SEQUENCE_LENGTH = dna_client.SEQUENCE_LENGTH_1MB
EPS = 1e-6
MIN_ABS_G_EFFECT = 1e-4
STRONG_QUANTILE = 0.90
LOCUS_FLANK_BP = 2_000
REF_QC_TOL = 1e-6
PRIMARY_SCOPE = "native_hematopoietic"

# Ventanas nominales recomendadas para outputs de cromatina:
# 501 bp para DNase/TF y 2001 bp para histonas.
MASK_HALF_WIDTH = {
    "DNASE": 250,
    "CHIP_TF": 250,
    "CHIP_HISTONE": 1000,
}

OUTPUT_SPECS = {
    "DNASE": (dna_client.OutputType.DNASE, "dnase"),
    "CHIP_HISTONE": (dna_client.OutputType.CHIP_HISTONE, "chip_histone"),
    "CHIP_TF": (dna_client.OutputType.CHIP_TF, "chip_tf"),
}

# Selección amplia y reproducible de sangre/hematopoyesis. Se audita en CSV antes
# de consumir la API. Las líneas hematológicas se incluyen en la sensibilidad,
# pero se excluyen del scope primario "native_hematopoietic".
HEMATOPOIETIC_REGEX = re.compile(
    r"blood|bone marrow|marrow|hematop|haematop|leuk|lymph|monocyt|macrophage|"
    r"neutroph|eosinoph|basophil|granulocyt|erythr|megakaryo|platelet|mast cell|"
    r"dendritic|\bb cell\b|\bt cell\b|natural killer|\bnk cell\b|plasma cell|"
    r"spleen|thymus|cd34|k562|gm12878|thp-?1|hl-?60|jurkat|kasumi|u937|nb4|"
    r"molm|nalm|reh cell|mm\.1s",
    flags=re.IGNORECASE,
)
NATIVE_TYPE_EXCLUDE = re.compile(
    r"cell[_ ]?line|in[_ ]?vitro|organoid|induced|immortal",
    flags=re.IGNORECASE,
)
CELL_LINE_NAME_REGEX = re.compile(
    r"k562|gm12878|thp-?1|hl-?60|jurkat|kasumi|u937|nb4|molm|nalm|reh cell|mm\.1s",
    flags=re.IGNORECASE,
)

# Opcional: lista explícita de CURIEs. Vacía = selección automática por regex.
ONTOLOGY_TERMS_OVERRIDE = ['CL:0000236', 'CL:0000623', 'CL:0000624', 'CL:0000625', 'CL:0000765', 'CL:0000775', 'CL:0000787', 'CL:0000788', 'CL:0000792', 'CL:0000823', 'CL:0000837', 'CL:0000862', 'CL:0000863', 'CL:0000895', 'CL:0000897', 'CL:0000900', 'CL:0000904', 'CL:0000905', 'CL:0000907', 'CL:0000909', 'CL:0000913', 'CL:0001044', 'CL:0001054', 'CL:0001059', 'CL:0010001', 'CL:2000001', 'CL:2000010', 'CL:2000041', 'CLO:0007045', 'EFO:0002067', 'EFO:0002784', 'EFO:0002793', 'EFO:0002798', 'EFO:0005724', 'UBERON:0002106', 'UBERON:0002370']

from google.colab import drive
drive.mount('/content/drive')
BASE_OUT = Path('/content/drive/MyDrive/SAMD9L_ventana_fija_54/cromatina')
BASE_OUT.mkdir(parents=True, exist_ok=True)

print("✅ Configuración cargada")

In [ ]:
manifest_path=BASE_OUT/'pares.csv'
manifest_path.write_text('Somatic_Event_ID,patient,background,germline_U1,germline_protein,g_variant_id,g_chrom,g_pos,g_ref,g_alt,somatic_ID,somatic_protein,somatic_cDNA,somatic_variant_class,s_variant_id,s_chrom,s_pos,s_ref,s_alt,pair_span_bp,contains_indel,overlap_check,priority_R589Ter\nSGR001,STJ_P012,I891T,INT003,I891T,chr7:93133300:A>G,chr7,93133300,A,G,SOMVAR015,p.K768X,c.2302A>T,nonsense,chr7:93133670:T>A,chr7,93133670,T,A,370,False,False,False\nSGR002,STJ_P017,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR002,p.R986L,c.2957G>T,missense_or_snv,chr7:93133015:C>A,chr7,93133015,C,A,1,False,False,False\nSGR003,STJ_P017,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR007,p.L410RfsX8,c.1204_1208del,frameshift,chr7:93134763:TATGAG>T,chr7,93134763,TATGAG,T,1747,True,False,False\nSGR004,STJ_P031,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR032,p.A1113T,c.3337G>A,missense_or_snv,chr7:93132635:C>T,chr7,93132635,C,T,381,False,False,False\nSGR005,STJ_P039,H880Q,INT002,H880Q,chr7:93133332:G>T,chr7,93133332,G,T,SOMVAR040,p.S1317RfsX21,c.3951_3955delTAAAG,frameshift,chr7:93132016:TCTTTA>T,chr7,93132016,TCTTTA,T,1316,True,False,False\nSGR006,STJ_P044,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR037,p.Q1183*,c.3547C>T,nonsense,chr7:93132425:G>A,chr7,93132425,G,A,295,False,False,False\nSGR007,STJ_P044,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR033,p.T1122S,c.3364A>T,missense_or_snv,chr7:93132608:T>A,chr7,93132608,T,A,478,False,False,False\nSGR008,STJ_P044,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR034,p.1122_1124del,c.3365_3370del,inframe_indel,chr7:93132601:CCTAGTG>C,chr7,93132601,CCTAGTG,C,471,True,False,False\nSGR009,STJ_P044,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR041,p.S1379P,c.4189T>C,missense_or_snv,chr7:93131783:A>G,chr7,93131783,A,G,347,False,False,False\nSGR010,STJ_P045,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR002,p.R986L,c.2957G>T,missense_or_snv,chr7:93133015:C>A,chr7,93133015,C,A,885,False,False,False\nSGR011,STJ_P045,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR003,p.1152_1153del,c.3456_3458del,inframe_indel,chr7:93132513:TAGG>T,chr7,93132513,TAGG,T,383,True,False,False\nSGR012,STJ_P046,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR006,p.V361fs,c.1082delT,frameshift,chr7:93134889:TA>T,chr7,93134889,TA,T,2759,True,False,False\nSGR013,STJ_P048,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR044,p.V19E,c.56T>A,missense_or_snv,chr7:93135916:A>T,chr7,93135916,A,T,3786,False,False,False\nSGR014,STJ_P048,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR011,p.E562K,c.1684G>A,missense_or_snv,chr7:93134288:C>T,chr7,93134288,C,T,2158,False,False,False\nSGR015,STJ_P048,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR017,p.E776fs,c.2326dupG,frameshift,chr7:93133645:T>TC,chr7,93133645,T,TC,1515,True,False,False\nSGR017,STJ_P049,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR001,p.R589*,c.1765C>T,nonsense,chr7:93134207:G>A,chr7,93134207,G,A,2077,False,False,True\nSGR018,STJ_P050,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR008,p.L409fs,c.1225delC,frameshift,chr7:93134746:AG>A,chr7,93134746,AG,A,3309,True,False,False\nSGR019,STJ_P055,V1512M,INT018,V1512M,chr7:93131438:C>T,chr7,93131438,C,T,SOMVAR004,p.R1188*,c.3562C>T,nonsense,chr7:93132410:G>A,chr7,93132410,G,A,972,False,False,False\nSGR020,STJ_P056,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR030,p.F1092L,c.3276C>G,missense_or_snv,chr7:93132696:G>C,chr7,93132696,G,C,320,False,False,False\nSGR021,STJ_P057,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR018,p.Q780*,c.2338C>T,nonsense,chr7:93133634:G>A,chr7,93133634,G,A,2197,False,False,False\nSGR022,STJ_P058,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR014,p.R70C,c.208C>T,missense_or_snv,chr7:93135764:G>A,chr7,93135764,G,A,3634,False,False,False\nSGR023,STJ_P058,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR047,p.A280T,c.838G>A,missense_or_snv,chr7:93135134:C>T,chr7,93135134,C,T,3004,False,False,False\nSGR024,STJ_P058,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR005,p.R359Q,c.1076G>A,missense_or_snv,chr7:93134896:C>T,chr7,93134896,C,T,2766,False,False,False\nSGR026,STJ_P058,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR029,p.F1092V,c.3274T>G,missense_or_snv,chr7:93132698:A>C,chr7,93132698,A,C,568,False,False,False\nSGR027,STJ_P058,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR035,p.D1171N,c.3511G>A,missense_or_snv,chr7:93132461:C>T,chr7,93132461,C,T,331,False,False,False\nSGR028,STJ_P060,S1473N,INT021,S1473N,chr7:93131554:C>T,chr7,93131554,C,T,SOMVAR012,p.S608TfsX6,c.1821_1822dupCA,frameshift,chr7:93134149:C>CTG,chr7,93134149,C,CTG,2595,True,False,False\nSGR029,STJ_P061,S1473N,INT021,S1473N,chr7:93131554:C>T,chr7,93131554,C,T,SOMVAR010,p.E461VfsX43,c.1382_1385delAAAG,frameshift,chr7:93134586:ACTTT>A,chr7,93134586,ACTTT,A,3032,True,False,False\nSGR030,STJ_P068,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR024,p.Lys1008Ile,c.3023A>T,missense_or_snv,chr7:93132949:T>A,chr7,93132949,T,A,1502,False,False,False\nSGR031,STJ_P068,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR036,p.Trp1180*,c.3540G>A,nonsense,chr7:93132432:C>T,chr7,93132432,C,T,985,False,False,False\nSGR032,STJ_P077,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR013,p.W695C,c.2085G>C,missense_or_snv,chr7:93133887:C>G,chr7,93133887,C,G,872,False,False,False\nSGR033,STJ_P084,P1349L,INT037,P1349L,chr7:93131926:G>A,chr7,93131926,G,A,SOMVAR046,p.K257X,c.768dupT,nonsense,chr7:93135203:T>TA,chr7,93135203,T,TA,3277,True,False,False\nSGR034,STJ_P085,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR028,p.R1077Q,c.3230G>A,missense_or_snv,chr7:93132742:C>T,chr7,93132742,C,T,274,False,False,False\nSGR035,STJ_P086,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR001,p.R589X,c.1765C>T,nonsense,chr7:93134207:G>A,chr7,93134207,G,A,1191,False,False,True\nSGR036,STJ_P093,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR020,p.Arg828*,c.2482C>T,nonsense,chr7:93133490:G>A,chr7,93133490,G,A,475,False,False,False\nSGR037,STJ_P093,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR026,p.Glu1060*,c.3178G>T,nonsense,chr7:93132794:C>A,chr7,93132794,C,A,221,False,False,False\nSGR038,STJ_P094,T936I,INT042,T936I,chr7:93133165:G>A,chr7,93133165,G,A,SOMVAR001,p.Arg589*,c.1765C>T,nonsense,chr7:93134207:G>A,chr7,93134207,G,A,1042,False,False,True\nSGR039,STJ_P096,K770E,INT040,K770E,chr7:93133664:T>C,chr7,93133664,T,C,SOMVAR001,p.R589X,c.1765C>T,nonsense,chr7:93134207:G>A,chr7,93134207,G,A,543,False,False,True\nSGR040,STJ_P099,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR027,p.R1077*,c.3229C>T,nonsense,chr7:93132743:G>A,chr7,93132743,G,A,314,False,False,False\nSGR041,STJ_P099,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR003,p.L1153del,c.3456_3458del,inframe_indel,chr7:93132513:TAGG>T,chr7,93132513,TAGG,T,544,True,False,False\nSGR042,STJ_P103,I876T,INT048,I876T,chr7:93133345:A>G,chr7,93133345,A,G,SOMVAR009,p.G454R,c.1360G>A,missense_or_snv,chr7:93134612:C>T,chr7,93134612,C,T,1267,False,False,False\nSGR043,STJ_P105,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR043,p.R1524H,c.4571G>A,missense_or_snv,chr7:93131401:C>T,chr7,93131401,C,T,1614,False,False,False\nSGR045,STJ_P114,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR031,p.K1095E,c.3283A>G,missense_or_snv,chr7:93132689:T>C,chr7,93132689,T,C,327,False,False,False\nSGR046,STJ_P129,S888F,INT060,S888F,chr7:93133309:G>A,chr7,93133309,G,A,SOMVAR039,p.Phe1307Ser,c.3920T>C,missense_or_snv,chr7:93132052:A>G,chr7,93132052,A,G,1257,False,False,False\nSGR047,STJ_P089,T1474P,INT039,T1474P,chr7:93131552:T>G,chr7,93131552,T,G,SOMVAR001,p.Arg589Ter,c.1765C>T,nonsense,chr7:93134207:G>A,chr7,93134207,G,A,2655,False,False,True\nSGR048,PUB_37584291_P4_B063,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR045,p.Cys228Tyr,c.683G>A,missense_or_snv,chr7:93135289:C>T,chr7,93135289,C,T,2274,False,False,False\nSGR049,PUB_37584291_P4_B063,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR023,p.Tyr900Cys,,missense_or_snv,chr7:93133273:T>C,chr7,93133273,T,C,258,False,False,False\nSGR050,PUB_37584291_P4_B063,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR004,p.Arg1188Ter,c.3562C>T,nonsense,chr7:93132410:G>A,chr7,93132410,G,A,605,False,False,False\nSGR051,PUB_37584291_P6_CZ132,R1281W,INT105,R1281W,chr7:93132131:T>A,chr7,93132131,T,A,SOMVAR042,p.Gln1409ThrfsTer49,c.4224dupA,frameshift,chr7:93131747:G>GT,chr7,93131747,G,GT,384,True,False,False\nSGR052,PUB_37584291_P7_A146,D1034Y,INT014,D1034Y,chr7:93132872:C>A,chr7,93132872,C,A,SOMVAR019,p.Tyr795Ter,c.2385C>A,nonsense,chr7:93133587:G>T,chr7,93133587,G,T,715,False,False,False\nSGR053,LIT_LIT009,G1510A,INT092,G1510A,chr7:93131443:C>G,chr7,93131443,C,G,SOMVAR038,p.His1218ProfsTer5,c.3650dup,frameshift,chr7:93132321:G>GA,chr7,93132321,G,GA,878,True,False,False\nSGR054,LIT_LIT010,S1550P,INT013,S1550P,chr7:93131324:A>G,chr7,93131324,A,G,SOMVAR022,p.Ala856Ter,c.2566_2569del,nonsense,chr7:93133402:AGTGC>A,chr7,93133402,AGTGC,A,2078,True,False,False\nSGR055,LIT_LIT008,G1510A,INT092,G1510A,chr7:93131443:C>G,chr7,93131443,C,G,SOMVAR048,p.Lys329ThrfsTer3,c.985_986insC,frameshift,chr7:93134986:T>TG,chr7,93134986,T,TG,3543,True,False,False\nSGR056,STJ_P050,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR049,p.Lys1265Asn,c.3795A>T,missense_or_snv,chr7:93132177:T>A,chr7,93132177,T,A,740,False,False,False\nSGR057,STJ_P126,K1532N,INT058,K1532N,chr7:93131376:C>A,chr7,93131376,C,A,SOMVAR050,p.Glu685del,c.2053_2055del,inframe_deletion,chr7:93133916:GTTC>G,chr7,93133916,GTTC,G,2540,True,False,False\nCTRL143,computational_control,I891T,INT003,I891T,chr7:93133300:A>G,chr7,93133300,A,G,SOMVAR018,p.Q780*,c.2338C>T,nonsense,chr7:93133634:G>A,chr7,93133634,G,A,334,False,False,False\nCTRL142,computational_control,I891T,INT003,I891T,chr7:93133300:A>G,chr7,93133300,A,G,SOMVAR019,p.Tyr795Ter,c.2385C>A,nonsense,chr7:93133587:G>T,chr7,93133587,G,T,287,False,False,False\nCTRL141,computational_control,I891T,INT003,I891T,chr7:93133300:A>G,chr7,93133300,A,G,SOMVAR020,p.Arg828*,c.2482C>T,nonsense,chr7:93133490:G>A,chr7,93133490,G,A,190,False,False,False\nCTRL145,computational_control,I891T,INT003,I891T,chr7:93133300:A>G,chr7,93133300,A,G,SOMVAR001,p.R589*,c.1765C>T,nonsense,chr7:93134207:G>A,chr7,93134207,G,A,907,False,False,False\nCTRL144,computational_control,I891T,INT003,I891T,chr7:93133300:A>G,chr7,93133300,A,G,SOMVAR017,p.E776fs,c.2326dupG,frameshift,chr7:93133645:T>TC,chr7,93133645,T,TC,345,True,False,False\nCTRL115,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR024,p.Lys1008Ile,c.3023A>T,missense_or_snv,chr7:93132949:T>A,chr7,93132949,T,A,67,False,False,False\nCTRL114,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,chr7:93132814:G>A,p.T1053I,,missense,chr7:93132814:G>A,chr7,93132814,G,A,202,False,False,False\nCTRL116,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR023,p.Tyr900Cys,,missense_or_snv,chr7:93133273:T>C,chr7,93133273,T,C,257,False,False,False\nCTRL113,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR029,p.F1092V,c.3274T>G,missense_or_snv,chr7:93132698:A>C,chr7,93132698,A,C,318,False,False,False\nCTRL112,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR033,p.T1122S,c.3364A>T,missense_or_snv,chr7:93132608:T>A,chr7,93132608,T,A,408,False,False,False\nCTRL123,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR008,p.L409fs,c.1225delC,frameshift,chr7:93134746:AG>A,chr7,93134746,AG,A,1730,True,False,False\nCTRL124,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR006,p.V361fs,c.1082delT,frameshift,chr7:93134889:TA>T,chr7,93134889,TA,T,1873,True,False,False\nCTRL122,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR010,p.E461VfsX43,c.1382_1385delAAAG,frameshift,chr7:93134586:ACTTT>A,chr7,93134586,ACTTT,A,1570,True,False,False\nCTRL125,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR048,p.Lys329ThrfsTer3,c.985_986insC,frameshift,chr7:93134986:T>TG,chr7,93134986,T,TG,1970,True,False,False\nCTRL121,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR012,p.S608TfsX6,c.1821_1822dupCA,frameshift,chr7:93134149:C>CTG,chr7,93134149,C,CTG,1133,True,False,False\nCTRL111,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR035,p.D1171N,c.3511G>A,missense_or_snv,chr7:93132461:C>T,chr7,93132461,C,T,555,False,False,False\nCTRL110,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR039,p.Phe1307Ser,c.3920T>C,missense_or_snv,chr7:93132052:A>G,chr7,93132052,A,G,964,False,False,False\nCTRL151,computational_control,H880Q,INT002,H880Q,chr7:93133332:G>T,chr7,93133332,G,T,SOMVAR042,p.Gln1409ThrfsTer49,c.4224dupA,frameshift,chr7:93131747:G>GT,chr7,93131747,G,GT,1585,True,False,False\nCTRL152,computational_control,H880Q,INT002,H880Q,chr7:93133332:G>T,chr7,93133332,G,T,SOMVAR038,p.His1218ProfsTer5,c.3650dup,frameshift,chr7:93132321:G>GA,chr7,93132321,G,GA,1011,True,False,False\nCTRL153,computational_control,H880Q,INT002,H880Q,chr7:93133332:G>T,chr7,93133332,G,T,SOMVAR004,p.R1188*,c.3562C>T,nonsense,chr7:93132410:G>A,chr7,93132410,G,A,922,False,False,False\nCTRL154,computational_control,H880Q,INT002,H880Q,chr7:93133332:G>T,chr7,93133332,G,T,SOMVAR037,p.Q1183*,c.3547C>T,nonsense,chr7:93132425:G>A,chr7,93132425,G,A,907,False,False,False\nCTRL155,computational_control,H880Q,INT002,H880Q,chr7:93133332:G>T,chr7,93133332,G,T,SOMVAR036,p.Trp1180*,c.3540G>A,nonsense,chr7:93132432:C>T,chr7,93132432,C,T,900,False,False,False\nCTRL057,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR036,p.Trp1180*,c.3540G>A,nonsense,chr7:93132432:C>T,chr7,93132432,C,T,302,False,False,False\nCTRL062,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR027,p.R1077*,c.3229C>T,nonsense,chr7:93132743:G>A,chr7,93132743,G,A,613,False,False,False\nCTRL063,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR026,p.Glu1060*,c.3178G>T,nonsense,chr7:93132794:C>A,chr7,93132794,C,A,664,False,False,False\nCTRL066,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR020,p.Arg828*,c.2482C>T,nonsense,chr7:93133490:G>A,chr7,93133490,G,A,1360,False,False,False\nCTRL067,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR019,p.Tyr795Ter,c.2385C>A,nonsense,chr7:93133587:G>T,chr7,93133587,G,T,1457,False,False,False\nCTRL058,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR032,p.A1113T,c.3337G>A,missense_or_snv,chr7:93132635:C>T,chr7,93132635,C,T,505,False,False,False\nCTRL059,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR031,p.K1095E,c.3283A>G,missense_or_snv,chr7:93132689:T>C,chr7,93132689,T,C,559,False,False,False\nCTRL060,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR030,p.F1092L,c.3276C>G,missense_or_snv,chr7:93132696:G>C,chr7,93132696,G,C,566,False,False,False\nCTRL061,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR028,p.R1077Q,c.3230G>A,missense_or_snv,chr7:93132742:C>T,chr7,93132742,C,T,612,False,False,False\nCTRL064,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR024,p.Lys1008Ile,c.3023A>T,missense_or_snv,chr7:93132949:T>A,chr7,93132949,T,A,819,False,False,False\nCTRL054,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR043,p.R1524H,c.4571G>A,missense_or_snv,chr7:93131401:C>T,chr7,93131401,C,T,729,False,False,False\nCTRL055,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR039,p.Phe1307Ser,c.3920T>C,missense_or_snv,chr7:93132052:A>G,chr7,93132052,A,G,78,False,False,False\nCTRL065,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR023,p.Tyr900Cys,,missense_or_snv,chr7:93133273:T>C,chr7,93133273,T,C,1143,False,False,False\nCTRL076,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR048,p.Lys329ThrfsTer3,c.985_986insC,frameshift,chr7:93134986:T>TG,chr7,93134986,T,TG,2856,True,False,False\nCTRL075,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR007,p.L410RfsX8,c.1204_1208del,frameshift,chr7:93134763:TATGAG>T,chr7,93134763,TATGAG,T,2633,True,False,False\nCTRL074,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR008,p.L409fs,c.1225delC,frameshift,chr7:93134746:AG>A,chr7,93134746,AG,A,2616,True,False,False\nCTRL072,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR010,p.E461VfsX43,c.1382_1385delAAAG,frameshift,chr7:93134586:ACTTT>A,chr7,93134586,ACTTT,A,2456,True,False,False\nCTRL071,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR012,p.S608TfsX6,c.1821_1822dupCA,frameshift,chr7:93134149:C>CTG,chr7,93134149,C,CTG,2019,True,False,False\nCTRL077,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR045,p.Cys228Tyr,c.683G>A,missense_or_snv,chr7:93135289:C>T,chr7,93135289,C,T,3159,False,False,False\nCTRL073,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR009,p.G454R,c.1360G>A,missense_or_snv,chr7:93134612:C>T,chr7,93134612,C,T,2482,False,False,False\nCTRL070,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR013,p.W695C,c.2085G>C,missense_or_snv,chr7:93133887:C>G,chr7,93133887,C,G,1757,False,False,False\nCTRL056,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR004,p.R1188*,c.3562C>T,nonsense,chr7:93132410:G>A,chr7,93132410,G,A,280,False,False,False\nCTRL069,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR015,p.K768X,c.2302A>T,nonsense,chr7:93133670:T>A,chr7,93133670,T,A,1540,False,False,False\nCTRL068,computational_control,R1281K,INT006,R1281K,chr7:93132130:C>T,chr7,93132130,C,T,SOMVAR018,p.Q780*,c.2338C>T,nonsense,chr7:93133634:G>A,chr7,93133634,G,A,1504,False,False,False\nCTRL013,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR007,p.L410RfsX8,c.1204_1208del,frameshift,chr7:93134763:TATGAG>T,chr7,93134763,TATGAG,T,3326,True,False,False\nCTRL014,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR006,p.V361fs,c.1082delT,frameshift,chr7:93134889:TA>T,chr7,93134889,TA,T,3452,True,False,False\nCTRL012,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR010,p.E461VfsX43,c.1382_1385delAAAG,frameshift,chr7:93134586:ACTTT>A,chr7,93134586,ACTTT,A,3149,True,False,False\nCTRL015,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR048,p.Lys329ThrfsTer3,c.985_986insC,frameshift,chr7:93134986:T>TG,chr7,93134986,T,TG,3549,True,False,False\nCTRL011,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR012,p.S608TfsX6,c.1821_1822dupCA,frameshift,chr7:93134149:C>CTG,chr7,93134149,C,CTG,2712,True,False,False\nCTRL020,computational_control,V1512M,INT018,V1512M,chr7:93131438:C>T,chr7,93131438,C,T,SOMVAR001,p.R589*,c.1765C>T,nonsense,chr7:93134207:G>A,chr7,93134207,G,A,2769,False,False,False\nCTRL016,computational_control,V1512M,INT018,V1512M,chr7:93131438:C>T,chr7,93131438,C,T,SOMVAR037,p.Q1183*,c.3547C>T,nonsense,chr7:93132425:G>A,chr7,93132425,G,A,987,False,False,False\nCTRL017,computational_control,V1512M,INT018,V1512M,chr7:93131438:C>T,chr7,93131438,C,T,SOMVAR036,p.Trp1180*,c.3540G>A,nonsense,chr7:93132432:C>T,chr7,93132432,C,T,994,False,False,False\nCTRL018,computational_control,V1512M,INT018,V1512M,chr7:93131438:C>T,chr7,93131438,C,T,SOMVAR027,p.R1077*,c.3229C>T,nonsense,chr7:93132743:G>A,chr7,93132743,G,A,1305,False,False,False\nCTRL019,computational_control,V1512M,INT018,V1512M,chr7:93131438:C>T,chr7,93131438,C,T,SOMVAR026,p.Glu1060*,c.3178G>T,nonsense,chr7:93132794:C>A,chr7,93132794,C,A,1356,False,False,False\nCTRL010,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR015,p.K768X,c.2302A>T,nonsense,chr7:93133670:T>A,chr7,93133670,T,A,2233,False,False,False\nCTRL009,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR019,p.Tyr795Ter,c.2385C>A,nonsense,chr7:93133587:G>T,chr7,93133587,G,T,2150,False,False,False\nCTRL008,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR020,p.Arg828*,c.2482C>T,nonsense,chr7:93133490:G>A,chr7,93133490,G,A,2053,False,False,False\nCTRL007,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR026,p.Glu1060*,c.3178G>T,nonsense,chr7:93132794:C>A,chr7,93132794,C,A,1357,False,False,False\nCTRL006,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR027,p.R1077*,c.3229C>T,nonsense,chr7:93132743:G>A,chr7,93132743,G,A,1306,False,False,False\nCTRL045,computational_control,S1473N,INT021,S1473N,chr7:93131554:C>T,chr7,93131554,C,T,SOMVAR008,p.L409fs,c.1225delC,frameshift,chr7:93134746:AG>A,chr7,93134746,AG,A,3192,True,False,False\nCTRL046,computational_control,S1473N,INT021,S1473N,chr7:93131554:C>T,chr7,93131554,C,T,SOMVAR007,p.L410RfsX8,c.1204_1208del,frameshift,chr7:93134763:TATGAG>T,chr7,93134763,TATGAG,T,3209,True,False,False\nCTRL044,computational_control,S1473N,INT021,S1473N,chr7:93131554:C>T,chr7,93131554,C,T,SOMVAR017,p.E776fs,c.2326dupG,frameshift,chr7:93133645:T>TC,chr7,93133645,T,TC,2091,True,False,False\nCTRL047,computational_control,S1473N,INT021,S1473N,chr7:93131554:C>T,chr7,93131554,C,T,SOMVAR006,p.V361fs,c.1082delT,frameshift,chr7:93134889:TA>T,chr7,93134889,TA,T,3335,True,False,False\nCTRL048,computational_control,S1473N,INT021,S1473N,chr7:93131554:C>T,chr7,93131554,C,T,SOMVAR048,p.Lys329ThrfsTer3,c.985_986insC,frameshift,chr7:93134986:T>TG,chr7,93134986,T,TG,3432,True,False,False\nCTRL036,computational_control,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,chr7:93132814:G>A,p.T1053I,,missense,chr7:93132814:G>A,chr7,93132814,G,A,1367,False,False,False\nCTRL033,computational_control,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR028,p.R1077Q,c.3230G>A,missense_or_snv,chr7:93132742:C>T,chr7,93132742,C,T,1295,False,False,False\nCTRL032,computational_control,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR029,p.F1092V,c.3274T>G,missense_or_snv,chr7:93132698:A>C,chr7,93132698,A,C,1251,False,False,False\nCTRL031,computational_control,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR030,p.F1092L,c.3276C>G,missense_or_snv,chr7:93132696:G>C,chr7,93132696,G,C,1249,False,False,False\nCTRL030,computational_control,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR031,p.K1095E,c.3283A>G,missense_or_snv,chr7:93132689:T>C,chr7,93132689,T,C,1242,False,False,False\nCTRL029,computational_control,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR037,p.Q1183*,c.3547C>T,nonsense,chr7:93132425:G>A,chr7,93132425,G,A,978,False,False,False\nCTRL034,computational_control,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR027,p.R1077*,c.3229C>T,nonsense,chr7:93132743:G>A,chr7,93132743,G,A,1296,False,False,False\nCTRL035,computational_control,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR026,p.Glu1060*,c.3178G>T,nonsense,chr7:93132794:C>A,chr7,93132794,C,A,1347,False,False,False\nCTRL037,computational_control,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR020,p.Arg828*,c.2482C>T,nonsense,chr7:93133490:G>A,chr7,93133490,G,A,2043,False,False,False\nCTRL038,computational_control,S1509R,INT028,S1509R,chr7:93131447:T>G,chr7,93131447,T,G,SOMVAR019,p.Tyr795Ter,c.2385C>A,nonsense,chr7:93133587:G>T,chr7,93133587,G,T,2140,False,False,False\nCTRL104,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR011,p.E562K,c.1684G>A,missense_or_snv,chr7:93134288:C>T,chr7,93134288,C,T,1273,False,False,False\nCTRL105,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR009,p.G454R,c.1360G>A,missense_or_snv,chr7:93134612:C>T,chr7,93134612,C,T,1597,False,False,False\nCTRL099,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,chr7:93133445:G>A,p.R843W,,missense,chr7:93133445:G>A,chr7,93133445,G,A,430,False,False,False\nCTRL106,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR005,p.R359Q,c.1076G>A,missense_or_snv,chr7:93134896:C>T,chr7,93134896,C,T,1881,False,False,False\nCTRL107,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR047,p.A280T,c.838G>A,missense_or_snv,chr7:93135134:C>T,chr7,93135134,C,T,2119,False,False,False\nCTRL049,computational_control,P1349L,INT037,P1349L,chr7:93131926:G>A,chr7,93131926,G,A,SOMVAR022,p.Ala856Ter,c.2566_2569del,nonsense,chr7:93133402:AGTGC>A,chr7,93133402,AGTGC,A,1476,True,False,False\nCTRL053,computational_control,P1349L,INT037,P1349L,chr7:93131926:G>A,chr7,93131926,G,A,SOMVAR048,p.Lys329ThrfsTer3,c.985_986insC,frameshift,chr7:93134986:T>TG,chr7,93134986,T,TG,3060,True,False,False\nCTRL052,computational_control,P1349L,INT037,P1349L,chr7:93131926:G>A,chr7,93131926,G,A,SOMVAR006,p.V361fs,c.1082delT,frameshift,chr7:93134889:TA>T,chr7,93134889,TA,T,2963,True,False,False\nCTRL051,computational_control,P1349L,INT037,P1349L,chr7:93131926:G>A,chr7,93131926,G,A,SOMVAR007,p.L410RfsX8,c.1204_1208del,frameshift,chr7:93134763:TATGAG>T,chr7,93134763,TATGAG,T,2837,True,False,False\nCTRL050,computational_control,P1349L,INT037,P1349L,chr7:93131926:G>A,chr7,93131926,G,A,SOMVAR008,p.L409fs,c.1225delC,frameshift,chr7:93134746:AG>A,chr7,93134746,AG,A,2820,True,False,False\nCTRL120,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR015,p.K768X,c.2302A>T,nonsense,chr7:93133670:T>A,chr7,93133670,T,A,654,False,False,False\nCTRL119,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR018,p.Q780*,c.2338C>T,nonsense,chr7:93133634:G>A,chr7,93133634,G,A,618,False,False,False\nCTRL118,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR019,p.Tyr795Ter,c.2385C>A,nonsense,chr7:93133587:G>T,chr7,93133587,G,T,571,False,False,False\nCTRL117,computational_control,R986C,INT004,R986C,chr7:93133016:G>A,chr7,93133016,G,A,SOMVAR020,p.Arg828*,c.2482C>T,nonsense,chr7:93133490:G>A,chr7,93133490,G,A,474,False,False,False\nCTRL100,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR019,p.Tyr795Ter,c.2385C>A,nonsense,chr7:93133587:G>T,chr7,93133587,G,T,572,False,False,False\nCTRL101,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR018,p.Q780*,c.2338C>T,nonsense,chr7:93133634:G>A,chr7,93133634,G,A,619,False,False,False\nCTRL102,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR015,p.K768X,c.2302A>T,nonsense,chr7:93133670:T>A,chr7,93133670,T,A,655,False,False,False\nCTRL103,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR001,p.R589*,c.1765C>T,nonsense,chr7:93134207:G>A,chr7,93134207,G,A,1192,False,False,False\nCTRL098,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR022,p.Ala856Ter,c.2566_2569del,nonsense,chr7:93133402:AGTGC>A,chr7,93133402,AGTGC,A,387,True,False,False\nCTRL097,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR027,p.R1077*,c.3229C>T,nonsense,chr7:93132743:G>A,chr7,93132743,G,A,272,False,False,False\nCTRL093,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR036,p.Trp1180*,c.3540G>A,nonsense,chr7:93132432:C>T,chr7,93132432,C,T,583,False,False,False\nCTRL092,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR037,p.Q1183*,c.3547C>T,nonsense,chr7:93132425:G>A,chr7,93132425,G,A,590,False,False,False\nCTRL091,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR038,p.His1218ProfsTer5,c.3650dup,frameshift,chr7:93132321:G>GA,chr7,93132321,G,GA,694,True,False,False\nCTRL089,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR040,p.S1317RfsX21,c.3951_3955delTAAAG,frameshift,chr7:93132016:TCTTTA>T,chr7,93132016,TCTTTA,T,999,True,False,False\nCTRL139,computational_control,T936I,INT042,T936I,chr7:93133165:G>A,chr7,93133165,G,A,SOMVAR015,p.K768X,c.2302A>T,nonsense,chr7:93133670:T>A,chr7,93133670,T,A,505,False,False,False\nCTRL138,computational_control,T936I,INT042,T936I,chr7:93133165:G>A,chr7,93133165,G,A,SOMVAR018,p.Q780*,c.2338C>T,nonsense,chr7:93133634:G>A,chr7,93133634,G,A,469,False,False,False\nCTRL137,computational_control,T936I,INT042,T936I,chr7:93133165:G>A,chr7,93133165,G,A,SOMVAR019,p.Tyr795Ter,c.2385C>A,nonsense,chr7:93133587:G>T,chr7,93133587,G,T,422,False,False,False\nCTRL136,computational_control,T936I,INT042,T936I,chr7:93133165:G>A,chr7,93133165,G,A,SOMVAR020,p.Arg828*,c.2482C>T,nonsense,chr7:93133490:G>A,chr7,93133490,G,A,325,False,False,False\nCTRL140,computational_control,T936I,INT042,T936I,chr7:93133165:G>A,chr7,93133165,G,A,SOMVAR012,p.S608TfsX6,c.1821_1822dupCA,frameshift,chr7:93134149:C>CTG,chr7,93134149,C,CTG,984,True,False,False\nCTRL161,computational_control,K770E,INT040,K770E,chr7:93133664:T>C,chr7,93133664,T,C,SOMVAR015,p.K768X,c.2302A>T,nonsense,chr7:93133670:T>A,chr7,93133670,T,A,6,False,False,False\nCTRL162,computational_control,K770E,INT040,K770E,chr7:93133664:T>C,chr7,93133664,T,C,SOMVAR012,p.S608TfsX6,c.1821_1822dupCA,frameshift,chr7:93134149:C>CTG,chr7,93134149,C,CTG,485,True,False,False\nCTRL163,computational_control,K770E,INT040,K770E,chr7:93133664:T>C,chr7,93133664,T,C,SOMVAR010,p.E461VfsX43,c.1382_1385delAAAG,frameshift,chr7:93134586:ACTTT>A,chr7,93134586,ACTTT,A,922,True,False,False\nCTRL164,computational_control,K770E,INT040,K770E,chr7:93133664:T>C,chr7,93133664,T,C,SOMVAR008,p.L409fs,c.1225delC,frameshift,chr7:93134746:AG>A,chr7,93134746,AG,A,1082,True,False,False\nCTRL165,computational_control,K770E,INT040,K770E,chr7:93133664:T>C,chr7,93133664,T,C,SOMVAR007,p.L410RfsX8,c.1204_1208del,frameshift,chr7:93134763:TATGAG>T,chr7,93134763,TATGAG,T,1099,True,False,False\nCTRL135,computational_control,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR026,p.Glu1060*,c.3178G>T,nonsense,chr7:93132794:C>A,chr7,93132794,C,A,263,False,False,False\nCTRL129,computational_control,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR036,p.Trp1180*,c.3540G>A,nonsense,chr7:93132432:C>T,chr7,93132432,C,T,625,False,False,False\nCTRL128,computational_control,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR037,p.Q1183*,c.3547C>T,nonsense,chr7:93132425:G>A,chr7,93132425,G,A,632,False,False,False\nCTRL127,computational_control,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR004,p.R1188*,c.3562C>T,nonsense,chr7:93132410:G>A,chr7,93132410,G,A,647,False,False,False\nCTRL126,computational_control,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR038,p.His1218ProfsTer5,c.3650dup,frameshift,chr7:93132321:G>GA,chr7,93132321,G,GA,736,True,False,False\nCTRL131,computational_control,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR034,p.1122_1124del,c.3365_3370del,inframe_indel,chr7:93132601:CCTAGTG>C,chr7,93132601,CCTAGTG,C,456,True,False,False\nCTRL130,computational_control,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR035,p.D1171N,c.3511G>A,missense_or_snv,chr7:93132461:C>T,chr7,93132461,C,T,596,False,False,False\nCTRL132,computational_control,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR033,p.T1122S,c.3364A>T,missense_or_snv,chr7:93132608:T>A,chr7,93132608,T,A,449,False,False,False\nCTRL133,computational_control,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR032,p.A1113T,c.3337G>A,missense_or_snv,chr7:93132635:C>T,chr7,93132635,C,T,422,False,False,False\nCTRL134,computational_control,I972T,INT045,I972T,chr7:93133057:A>G,chr7,93133057,A,G,SOMVAR031,p.K1095E,c.3283A>G,missense_or_snv,chr7:93132689:T>C,chr7,93132689,T,C,368,False,False,False\nCTRL157,computational_control,I876T,INT048,I876T,chr7:93133345:A>G,chr7,93133345,A,G,SOMVAR005,p.R359Q,c.1076G>A,missense_or_snv,chr7:93134896:C>T,chr7,93134896,C,T,1551,False,False,False\nCTRL156,computational_control,I876T,INT048,I876T,chr7:93133345:A>G,chr7,93133345,A,G,SOMVAR011,p.E562K,c.1684G>A,missense_or_snv,chr7:93134288:C>T,chr7,93134288,C,T,943,False,False,False\nCTRL158,computational_control,I876T,INT048,I876T,chr7:93133345:A>G,chr7,93133345,A,G,SOMVAR047,p.A280T,c.838G>A,missense_or_snv,chr7:93135134:C>T,chr7,93135134,C,T,1789,False,False,False\nCTRL159,computational_control,I876T,INT048,I876T,chr7:93133345:A>G,chr7,93133345,A,G,SOMVAR045,p.Cys228Tyr,c.683G>A,missense_or_snv,chr7:93135289:C>T,chr7,93135289,C,T,1944,False,False,False\nCTRL160,computational_control,I876T,INT048,I876T,chr7:93133345:A>G,chr7,93133345,A,G,SOMVAR014,p.R70C,c.208C>T,missense_or_snv,chr7:93135764:G>A,chr7,93135764,G,A,2419,False,False,False\nCTRL088,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR041,p.S1379P,c.4189T>C,missense_or_snv,chr7:93131783:A>G,chr7,93131783,A,G,1232,False,False,False\nCTRL090,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR039,p.Phe1307Ser,c.3920T>C,missense_or_snv,chr7:93132052:A>G,chr7,93132052,A,G,963,False,False,False\nCTRL094,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR035,p.D1171N,c.3511G>A,missense_or_snv,chr7:93132461:C>T,chr7,93132461,C,T,554,False,False,False\nCTRL095,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR033,p.T1122S,c.3364A>T,missense_or_snv,chr7:93132608:T>A,chr7,93132608,T,A,407,False,False,False\nCTRL096,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR032,p.A1113T,c.3337G>A,missense_or_snv,chr7:93132635:C>T,chr7,93132635,C,T,380,False,False,False\nCTRL147,computational_control,S888F,INT060,S888F,chr7:93133309:G>A,chr7,93133309,G,A,SOMVAR041,p.S1379P,c.4189T>C,missense_or_snv,chr7:93131783:A>G,chr7,93131783,A,G,1526,False,False,False\nCTRL148,computational_control,S888F,INT060,S888F,chr7:93133309:G>A,chr7,93133309,G,A,SOMVAR035,p.D1171N,c.3511G>A,missense_or_snv,chr7:93132461:C>T,chr7,93132461,C,T,848,False,False,False\nCTRL146,computational_control,S888F,INT060,S888F,chr7:93133309:G>A,chr7,93133309,G,A,SOMVAR043,p.R1524H,c.4571G>A,missense_or_snv,chr7:93131401:C>T,chr7,93131401,C,T,1908,False,False,False\nCTRL149,computational_control,S888F,INT060,S888F,chr7:93133309:G>A,chr7,93133309,G,A,SOMVAR033,p.T1122S,c.3364A>T,missense_or_snv,chr7:93132608:T>A,chr7,93132608,T,A,701,False,False,False\nCTRL150,computational_control,S888F,INT060,S888F,chr7:93133309:G>A,chr7,93133309,G,A,SOMVAR032,p.A1113T,c.3337G>A,missense_or_snv,chr7:93132635:C>T,chr7,93132635,C,T,674,False,False,False\nCTRL039,computational_control,T1474P,INT039,T1474P,chr7:93131552:T>G,chr7,93131552,T,G,SOMVAR004,p.R1188*,c.3562C>T,nonsense,chr7:93132410:G>A,chr7,93132410,G,A,858,False,False,False\nCTRL043,computational_control,T1474P,INT039,T1474P,chr7:93131552:T>G,chr7,93131552,T,G,SOMVAR015,p.K768X,c.2302A>T,nonsense,chr7:93133670:T>A,chr7,93133670,T,A,2118,False,False,False\nCTRL042,computational_control,T1474P,INT039,T1474P,chr7:93131552:T>G,chr7,93131552,T,G,SOMVAR018,p.Q780*,c.2338C>T,nonsense,chr7:93133634:G>A,chr7,93133634,G,A,2082,False,False,False\nCTRL041,computational_control,T1474P,INT039,T1474P,chr7:93131552:T>G,chr7,93131552,T,G,SOMVAR019,p.Tyr795Ter,c.2385C>A,nonsense,chr7:93133587:G>T,chr7,93133587,G,T,2035,False,False,False\nCTRL040,computational_control,T1474P,INT039,T1474P,chr7:93131552:T>G,chr7,93131552,T,G,SOMVAR020,p.Arg828*,c.2482C>T,nonsense,chr7:93133490:G>A,chr7,93133490,G,A,1938,False,False,False\nCTRL108,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR014,p.R70C,c.208C>T,missense_or_snv,chr7:93135764:G>A,chr7,93135764,G,A,2749,False,False,False\nCTRL109,computational_control,R986H,INT017,R986H,chr7:93133015:C>T,chr7,93133015,C,T,SOMVAR044,p.V19E,c.56T>A,missense_or_snv,chr7:93135916:A>T,chr7,93135916,A,T,2901,False,False,False\nCTRL078,computational_control,R1281W,INT105,R1281W,chr7:93132131:T>A,chr7,93132131,T,A,SOMVAR040,p.S1317RfsX21,c.3951_3955delTAAAG,frameshift,chr7:93132016:TCTTTA>T,chr7,93132016,TCTTTA,T,115,True,False,False\nCTRL079,computational_control,R1281W,INT105,R1281W,chr7:93132131:T>A,chr7,93132131,T,A,SOMVAR038,p.His1218ProfsTer5,c.3650dup,frameshift,chr7:93132321:G>GA,chr7,93132321,G,GA,190,True,False,False\nCTRL080,computational_control,R1281W,INT105,R1281W,chr7:93132131:T>A,chr7,93132131,T,A,SOMVAR017,p.E776fs,c.2326dupG,frameshift,chr7:93133645:T>TC,chr7,93133645,T,TC,1514,True,False,False\nCTRL081,computational_control,R1281W,INT105,R1281W,chr7:93132131:T>A,chr7,93132131,T,A,SOMVAR012,p.S608TfsX6,c.1821_1822dupCA,frameshift,chr7:93134149:C>CTG,chr7,93134149,C,CTG,2018,True,False,False\nCTRL082,computational_control,R1281W,INT105,R1281W,chr7:93132131:T>A,chr7,93132131,T,A,SOMVAR010,p.E461VfsX43,c.1382_1385delAAAG,frameshift,chr7:93134586:ACTTT>A,chr7,93134586,ACTTT,A,2455,True,False,False\nCTRL084,computational_control,D1034Y,INT014,D1034Y,chr7:93132872:C>A,chr7,93132872,C,A,SOMVAR018,p.Q780*,c.2338C>T,nonsense,chr7:93133634:G>A,chr7,93133634,G,A,762,False,False,False\nCTRL086,computational_control,D1034Y,INT014,D1034Y,chr7:93132872:C>A,chr7,93132872,C,A,SOMVAR015,p.K768X,c.2302A>T,nonsense,chr7:93133670:T>A,chr7,93133670,T,A,798,False,False,False\nCTRL083,computational_control,D1034Y,INT014,D1034Y,chr7:93132872:C>A,chr7,93132872,C,A,SOMVAR020,p.Arg828*,c.2482C>T,nonsense,chr7:93133490:G>A,chr7,93133490,G,A,618,False,False,False\nCTRL087,computational_control,D1034Y,INT014,D1034Y,chr7:93132872:C>A,chr7,93132872,C,A,SOMVAR001,p.R589*,c.1765C>T,nonsense,chr7:93134207:G>A,chr7,93134207,G,A,1335,False,False,False\nCTRL085,computational_control,D1034Y,INT014,D1034Y,chr7:93132872:C>A,chr7,93132872,C,A,SOMVAR017,p.E776fs,c.2326dupG,frameshift,chr7:93133645:T>TC,chr7,93133645,T,TC,773,True,False,False\nCTRL022,computational_control,G1510A,INT092,G1510A,chr7:93131443:C>G,chr7,93131443,C,G,SOMVAR040,p.S1317RfsX21,c.3951_3955delTAAAG,frameshift,chr7:93132016:TCTTTA>T,chr7,93132016,TCTTTA,T,573,True,False,False\nCTRL023,computational_control,G1510A,INT092,G1510A,chr7:93131443:C>G,chr7,93131443,C,G,SOMVAR017,p.E776fs,c.2326dupG,frameshift,chr7:93133645:T>TC,chr7,93133645,T,TC,2202,True,False,False\nCTRL021,computational_control,G1510A,INT092,G1510A,chr7:93131443:C>G,chr7,93131443,C,G,SOMVAR042,p.Gln1409ThrfsTer49,c.4224dupA,frameshift,chr7:93131747:G>GT,chr7,93131747,G,GT,304,True,False,False\nCTRL024,computational_control,G1510A,INT092,G1510A,chr7:93131443:C>G,chr7,93131443,C,G,SOMVAR012,p.S608TfsX6,c.1821_1822dupCA,frameshift,chr7:93134149:C>CTG,chr7,93134149,C,CTG,2706,True,False,False\nCTRL025,computational_control,G1510A,INT092,G1510A,chr7:93131443:C>G,chr7,93131443,C,G,SOMVAR010,p.E461VfsX43,c.1382_1385delAAAG,frameshift,chr7:93134586:ACTTT>A,chr7,93134586,ACTTT,A,3143,True,False,False\nCTRL005,computational_control,S1550P,INT013,S1550P,chr7:93131324:A>G,chr7,93131324,A,G,SOMVAR046,p.K257X,c.768dupT,nonsense,chr7:93135203:T>TA,chr7,93135203,T,TA,3879,True,False,False\nCTRL001,computational_control,S1550P,INT013,S1550P,chr7:93131324:A>G,chr7,93131324,A,G,SOMVAR017,p.E776fs,c.2326dupG,frameshift,chr7:93133645:T>TC,chr7,93133645,T,TC,2321,True,False,False\nCTRL002,computational_control,S1550P,INT013,S1550P,chr7:93131324:A>G,chr7,93131324,A,G,SOMVAR012,p.S608TfsX6,c.1821_1822dupCA,frameshift,chr7:93134149:C>CTG,chr7,93134149,C,CTG,2825,True,False,False\nCTRL003,computational_control,S1550P,INT013,S1550P,chr7:93131324:A>G,chr7,93131324,A,G,SOMVAR010,p.E461VfsX43,c.1382_1385delAAAG,frameshift,chr7:93134586:ACTTT>A,chr7,93134586,ACTTT,A,3262,True,False,False\nCTRL004,computational_control,S1550P,INT013,S1550P,chr7:93131324:A>G,chr7,93131324,A,G,SOMVAR008,p.L409fs,c.1225delC,frameshift,chr7:93134746:AG>A,chr7,93134746,AG,A,3422,True,False,False\nCTRL028,computational_control,G1510A,INT092,G1510A,chr7:93131443:C>G,chr7,93131443,C,G,SOMVAR006,p.V361fs,c.1082delT,frameshift,chr7:93134889:TA>T,chr7,93134889,TA,T,3446,True,False,False\nCTRL027,computational_control,G1510A,INT092,G1510A,chr7:93131443:C>G,chr7,93131443,C,G,SOMVAR007,p.L410RfsX8,c.1204_1208del,frameshift,chr7:93134763:TATGAG>T,chr7,93134763,TATGAG,T,3320,True,False,False\nCTRL026,computational_control,G1510A,INT092,G1510A,chr7:93131443:C>G,chr7,93131443,C,G,SOMVAR008,p.L409fs,c.1225delC,frameshift,chr7:93134746:AG>A,chr7,93134746,AG,A,3303,True,False,False\nCTRL_SGR056_1,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR039,p.Phe1307Ser,c.3920T>C,missense_or_snv,chr7:93132052:A>G,chr7,93132052,A,G,615,False,False,False\nCTRL_SGR056_2,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR035,p.D1171N,c.3511G>A,missense_or_snv,chr7:93132461:C>T,chr7,93132461,C,T,1024,False,False,False\nCTRL_SGR056_3,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR033,p.T1122S,c.3364A>T,missense_or_snv,chr7:93132608:T>A,chr7,93132608,T,A,1171,False,False,False\nCTRL_SGR056_4,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR032,p.A1113T,c.3337G>A,missense_or_snv,chr7:93132635:C>T,chr7,93132635,C,T,1198,False,False,False\nCTRL_SGR056_5,computational_control,V1512A,INT019,V1512A,chr7:93131437:A>G,chr7,93131437,A,G,SOMVAR031,p.K1095E,c.3283A>G,missense_or_snv,chr7:93132689:T>C,chr7,93132689,T,C,1252,False,False,False\nCTRL_SGR057_1,computational_control,K1532N,INT058,K1532N,chr7:93131376:C>A,chr7,93131376,C,A,SOMVAR034,p.1122_1124del,c.3365_3370del,inframe_indel,chr7:93132601:CCTAGTG>C,chr7,93132601,CCTAGTG,C,1225,True,False,False\nCTRL_SGR057_2,computational_control,K1532N,INT058,K1532N,chr7:93131376:C>A,chr7,93131376,C,A,SOMVAR003,p.1152_1153del,c.3456_3458del,inframe_indel,chr7:93132513:TAGG>T,chr7,93132513,TAGG,T,1137,True,False,False\nCTRL_SGR057_3,computational_control,K1532N,INT058,K1532N,chr7:93131376:C>A,chr7,93131376,C,A,SOMVAR013,p.W695C,c.2085G>C,missense_or_snv,chr7:93133887:C>G,chr7,93133887,C,G,2511,False,False,False\nCTRL_SGR057_4,computational_control,K1532N,INT058,K1532N,chr7:93131376:C>A,chr7,93131376,C,A,SOMVAR011,p.E562K,c.1684G>A,missense_or_snv,chr7:93134288:C>T,chr7,93134288,C,T,2912,False,False,False\nCTRL_SGR057_5,computational_control,K1532N,INT058,K1532N,chr7:93131376:C>A,chr7,93131376,C,A,chr7:93133445:G>A,p.R843W,,missense,chr7:93133445:G>A,chr7,93133445,G,A,2069,False,False,False\n')

In [ ]:
# 4) Validar manifiesto y fijar UNA ventana de 1 Mb para todo el estudio
manifest = pd.read_csv(manifest_path)

required = {
    "Somatic_Event_ID", "patient", "background",
    "g_chrom", "g_pos", "g_ref", "g_alt",
    "s_chrom", "s_pos", "s_ref", "s_alt",
}
missing = sorted(required - set(manifest.columns))
if missing:
    raise ValueError(f"Faltan columnas: {missing}")
if len(manifest) != 229:
    raise ValueError(f"Se esperaban 229 configuraciones, hay {len(manifest)}")
if manifest["Somatic_Event_ID"].duplicated().any():
    raise ValueError("Somatic_Event_ID no es único")
if set(manifest["g_chrom"]) != {"chr7"} or set(manifest["s_chrom"]) != {"chr7"}:
    raise ValueError("Se esperaban exclusivamente coordenadas chr7/GRCh38")

starts0 = pd.concat([
    manifest["g_pos"].astype(int) - 1,
    manifest["s_pos"].astype(int) - 1,
])
ends0 = pd.concat([
    manifest["g_pos"].astype(int) - 1 + manifest["g_ref"].astype(str).str.len(),
    manifest["s_pos"].astype(int) - 1 + manifest["s_ref"].astype(str).str.len(),
])

GLOBAL_VARIANT_START0 = 93131323
GLOBAL_VARIANT_END0 = 93135916
center0 = (GLOBAL_VARIANT_START0 + GLOBAL_VARIANT_END0) // 2
FIXED_START0 = center0 - SEQUENCE_LENGTH // 2
FIXED_END0 = FIXED_START0 + SEQUENCE_LENGTH
FIXED_INTERVAL = genome.Interval("chr7", FIXED_START0, FIXED_END0)

LOCUS_START0 = GLOBAL_VARIANT_START0 - LOCUS_FLANK_BP
LOCUS_END0 = GLOBAL_VARIANT_END0 + LOCUS_FLANK_BP
if not (FIXED_START0 <= LOCUS_START0 < LOCUS_END0 <= FIXED_END0):
    raise ValueError("La máscara del locus no cabe en la ventana fija")

print("Pares:", len(manifest))
print("Fondos germinales:", manifest["background"].nunique())
print("Somáticas únicas:", manifest["somatic_ID"].nunique())
print("Ventana fija (0-based, half-open):", FIXED_INTERVAL)
print("Rango de variantes + 2 kb:", f"chr7:{LOCUS_START0}-{LOCUS_END0}")
display(manifest[[
    "Somatic_Event_ID", "patient", "background", "germline_protein",
    "somatic_protein", "contains_indel"
]].head(12))

In [ ]:
from getpass import getpass

api_key = getpass("Pega tu clave de AlphaGenome y pulsa Enter: ").strip()
if not api_key:
    raise ValueError("El campo está vacío. Ejecuta esta celda de nuevo.")
print("Clave recibida.")


In [ ]:
# 5) Conectar y seleccionar/auditar tracks hematopoyéticos ANTES de predecir
dna_model = dna_client.create(api_key)
metadata_all = dna_model.output_metadata(
    dna_client.Organism.HOMO_SAPIENS
).concatenate().copy()

def output_type_name(value):
    return value.name if hasattr(value, "name") else str(value).rsplit(".", 1)[-1]

metadata_all["output_type_name"] = metadata_all["output_type"].map(output_type_name)
for col in ["name", "biosample_name", "biosample_type", "ontology_curie"]:
    if col not in metadata_all.columns:
        metadata_all[col] = ""

text_for_filter = (
    metadata_all["biosample_name"].fillna("").astype(str) + " " +
    metadata_all["name"].fillna("").astype(str)
)
selected_metadata = metadata_all[
    metadata_all["output_type_name"].isin(OUTPUT_SPECS) &
    text_for_filter.str.contains(HEMATOPOIETIC_REGEX, na=False)
].copy()

if ONTOLOGY_TERMS_OVERRIDE:
    requested_terms = sorted(set(map(str, ONTOLOGY_TERMS_OVERRIDE)))
    selected_metadata = selected_metadata[
        selected_metadata["ontology_curie"].astype(str).isin(requested_terms)
    ].copy()
else:
    requested_terms = sorted(
        selected_metadata["ontology_curie"].dropna().astype(str).unique().tolist()
    )

requested_terms = [x for x in requested_terms if x and x.lower() != "nan"]
if not requested_terms:
    raise RuntimeError("La selección no produjo términos ontológicos")
selected_metadata = selected_metadata[
    selected_metadata["ontology_curie"].astype(str).isin(requested_terms)
].copy()

type_text = selected_metadata["biosample_type"].fillna("").astype(str)
name_text = selected_metadata["biosample_name"].fillna("").astype(str)
selected_metadata["is_native_hematopoietic"] = ~(
    type_text.str.contains(NATIVE_TYPE_EXCLUDE, na=False) |
    name_text.str.contains(CELL_LINE_NAME_REGEX, na=False)
)

config_payload = {
    "alphagenome": "0.9.0",
    "fixed_interval": {
        "chromosome": "chr7", "start": FIXED_START0, "end": FIXED_END0
    },
    "outputs": list(OUTPUT_SPECS),
    "ontology_terms": requested_terms,
    "mask_half_width": MASK_HALF_WIDTH,
    "locus_mask": [LOCUS_START0, LOCUS_END0],
    "eps": EPS,
    "strong_quantile": STRONG_QUANTILE,
    "min_abs_g_effect": MIN_ABS_G_EFFECT,
    "effect_reference": "paired_reference_per_variant_call_v2",
}
CONFIG_HASH = hashlib.sha256(
    json.dumps(config_payload, sort_keys=True).encode()
).hexdigest()[:12]
assert CONFIG_HASH == 'c866a01d1ca4', 'Configuración distinta de la archivada'
manifest_hash = hashlib.sha256(manifest_path.read_bytes()).hexdigest()[:12]
OUT = BASE_OUT / CONFIG_HASH / manifest_hash
PAIR_DIR = OUT / "pair_metrics"
STATE_DIR = OUT / "state_summaries"
QC_DIR = OUT / "qc"
REF_CACHE = OUT / "grch38_cache"
FIG_DIR = OUT / "figures"
for directory in [PAIR_DIR, STATE_DIR, QC_DIR, REF_CACHE, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

(OUT / "analysis_config.json").write_text(
    json.dumps(config_payload, indent=2, sort_keys=True), encoding="utf-8"
)
manifest.to_csv(OUT / "SAMD9L_CIS_RESCUE_54_PAIRS_MANIFEST.csv", index=False)
selected_metadata.to_csv(OUT / "selected_hematopoietic_track_metadata.csv", index=False)

coverage = (
    selected_metadata.groupby(["output_type_name", "is_native_hematopoietic"])
    .agg(tracks=("name", "size"), ontology_terms=("ontology_curie", "nunique"))
    .reset_index()
)
print("✅ AlphaGenome conectado")
print("Términos ontológicos solicitados:", len(requested_terms))
print("Configuración:", CONFIG_HASH)
display(coverage)
display(selected_metadata[[
    c for c in ["output_type_name", "ontology_curie", "biosample_name",
                "biosample_type", "histone_mark", "transcription_factor",
                "is_native_hematopoietic"]
    if c in selected_metadata.columns
]].drop_duplicates().head(40))

In [ ]:
# 6) Funciones: haplotipo, máscaras fijas, agregación y checkpoints
METADATA_KEEP = [
    "name", "strand", "Assay title", "ontology_curie", "biosample_name",
    "biosample_type", "biosample_life_stage", "data_source", "endedness",
    "genetically_modified", "nonzero_mean", "histone_mark",
    "transcription_factor",
]

def fetch_grch38_region(chrom, start_1based, end_1based):
    """Descarga GRCh38 (NC_000007.14) y usa caché local reproducible."""
    if chrom != "chr7":
        raise ValueError(f"Cromosoma inesperado: {chrom}")
    start_1based, end_1based = int(start_1based), int(end_1based)
    cache = REF_CACHE / f"{chrom}_{start_1based}_{end_1based}.txt"
    if cache.exists():
        seq = cache.read_text(encoding="utf-8").strip().upper()
    else:
        params = {
            "db": "nuccore", "id": "NC_000007.14", "report": "fasta",
            "retmode": "text", "seq_start": start_1based,
            "seq_stop": end_1based, "strand": 1,
        }
        url = "https://www.ncbi.nlm.nih.gov/sviewer/viewer.fcgi?" + urlencode(params)
        last_error = None
        for attempt in range(API_RETRIES):
            try:
                req = Request(url, headers={"User-Agent": "SAMD9L-fixed-window-cis/1.0"})
                with urlopen(req, timeout=60) as response:
                    text = response.read().decode("utf-8")
                seq = "".join(
                    line.strip() for line in text.splitlines()
                    if not line.startswith(">")
                ).upper()
                cache.write_text(seq, encoding="utf-8")
                time.sleep(0.34)  # Evita ráfagas contra NCBI durante el preflight.
                break
            except Exception as exc:
                last_error = exc
                if attempt + 1 == API_RETRIES:
                    raise
                time.sleep(RETRY_BASE_SECONDS * (2 ** attempt))
        if last_error is not None and not cache.exists():
            raise last_error
    expected = end_1based - start_1based + 1
    if len(seq) != expected:
        raise RuntimeError(f"GRCh38 length mismatch: {len(seq)} != {expected}")
    return seq


def make_variant(chrom, pos, ref, alt):
    return genome.Variant(
        chromosome=str(chrom),
        position=int(pos),
        reference_bases=str(ref).upper(),
        alternate_bases=str(alt).upper(),
    )


def construct_combined_variant(germline, somatic):
    """Construye un único Variant que aplica ambas ediciones sobre GRCh38."""
    if germline.chromosome != somatic.chromosome:
        raise ValueError("Las variantes están en cromosomas distintos")
    variants = sorted([germline, somatic], key=lambda v: v.position)
    first, second = variants
    if first.position + len(first.reference_bases) - 1 >= second.position:
        raise ValueError("Variantes normalizadas solapantes")

    start = min(v.position for v in variants)
    end = max(v.position + len(v.reference_bases) - 1 for v in variants)
    ref_seq = fetch_grch38_region(germline.chromosome, start, end)

    for variant in variants:
        offset = variant.position - start
        observed = ref_seq[offset:offset + len(variant.reference_bases)]
        if observed != variant.reference_bases.upper():
            raise ValueError(
                f"REF mismatch {variant.chromosome}:{variant.position}: "
                f"GRCh38={observed}, manifiesto={variant.reference_bases}"
            )

    alt_seq = ref_seq
    # Orden inverso: una indel temprana no desplaza el offset de la segunda edición.
    for variant in sorted(variants, key=lambda v: v.position, reverse=True):
        offset = variant.position - start
        alt_seq = (
            alt_seq[:offset] + variant.alternate_bases.upper() +
            alt_seq[offset + len(variant.reference_bases):]
        )

    return genome.Variant(
        chromosome=germline.chromosome,
        position=start,
        reference_bases=ref_seq,
        alternate_bases=alt_seq,
    )


def pair_variants(row):
    germline = make_variant(row.g_chrom, row.g_pos, row.g_ref, row.g_alt)
    somatic = make_variant(row.s_chrom, row.s_pos, row.s_ref, row.s_alt)
    combined = construct_combined_variant(germline, somatic)
    for variant in [germline, somatic, combined]:
        start0 = variant.position - 1
        end0 = start0 + len(variant.reference_bases)
        if not (FIXED_START0 <= start0 < end0 <= FIXED_END0):
            raise ValueError(f"Variante fuera del intervalo fijo: {variant}")
    return germline, somatic, combined


def mask_segments(row, output_type):
    half = MASK_HALF_WIDTH[output_type]
    pair_union = []
    for prefix in ["g", "s"]:
        pos0 = int(row[f"{prefix}_pos"]) - 1
        ref_len = len(str(row[f"{prefix}_ref"]))
        pair_union.append((pos0 - half, pos0 + ref_len + half))
    return {
        "PRIMARY_PAIR_UNION": pair_union,
        "SAMD9L_VARIANT_RANGE_PLUS_2KB": [(LOCUS_START0, LOCUS_END0)],
        "FULL_1MB": [(FIXED_START0, FIXED_END0)],
    }


def bins_overlapping_segments(track_data, segments):
    values = np.asarray(track_data.values)
    if values.ndim != 2:
        raise ValueError(f"TrackData inesperado: shape={values.shape}")
    resolution = int(track_data.resolution)
    n_bins = values.shape[0]
    expected_bins = SEQUENCE_LENGTH // resolution
    if n_bins != expected_bins:
        raise ValueError(
            f"Número de bins inesperado: {n_bins} != {expected_bins} "
            f"(resolución={resolution})"
        )
    if track_data.interval is not None:
        td_interval = track_data.interval
        if (
            td_interval.chromosome != "chr7" or
            int(td_interval.start) != FIXED_START0 or
            int(td_interval.end) != FIXED_END0
        ):
            raise ValueError(f"TrackData no usa la ventana fija: {td_interval}")

    bin_start = FIXED_START0 + np.arange(n_bins, dtype=np.int64) * resolution
    bin_end = bin_start + resolution
    mask = np.zeros(n_bins, dtype=bool)
    for start0, end0 in segments:
        start0 = max(int(start0), FIXED_START0)
        end0 = min(int(end0), FIXED_END0)
        mask |= (bin_start < end0) & (bin_end > start0)
    if not mask.any():
        raise ValueError(f"Máscara vacía: {segments}")
    return mask


def safe_scalar(value):
    if value is None or isinstance(value, (str, int, float, bool, np.number)):
        return value
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return str(value)


def track_metadata(track_data):
    metadata = track_data.metadata.reset_index(drop=True).copy()
    keep = [col for col in METADATA_KEEP if col in metadata.columns]
    metadata = metadata[keep].copy()
    for col in metadata.columns:
        if metadata[col].dtype == "object":
            metadata[col] = metadata[col].map(safe_scalar)
    for col in ["name", "strand", "biosample_name", "biosample_type"]:
        if col not in metadata.columns:
            metadata[col] = ""
    metadata["track_index"] = np.arange(len(metadata), dtype=int)
    metadata["track_key"] = (
        metadata["name"].fillna("").astype(str) + "||" +
        metadata["strand"].fillna("").astype(str)
    )
    if metadata["track_key"].duplicated().any():
        raise ValueError("Track key duplicada (name, strand)")
    native_excluded = (
        metadata["biosample_type"].fillna("").astype(str)
        .str.contains(NATIVE_TYPE_EXCLUDE, na=False) |
        metadata["biosample_name"].fillna("").astype(str)
        .str.contains(CELL_LINE_NAME_REGEX, na=False)
    )
    metadata["is_native_hematopoietic"] = ~native_excluded
    return metadata


def summarise_output(output, state, row):
    frames = []
    for output_name, (_, attribute) in OUTPUT_SPECS.items():
        track_data = getattr(output, attribute)
        if track_data is None:
            continue
        metadata = track_metadata(track_data)
        values = np.asarray(track_data.values)
        for mask_id, segments in mask_segments(row, output_name).items():
            mask = bins_overlapping_segments(track_data, segments)
            # Evita copiar toda la matriz de DNase cuando la máscara es 1 Mb.
            if bool(mask.all()):
                signal_sum = values.sum(axis=0, dtype=np.float64)
            else:
                signal_sum = values[mask, :].sum(axis=0, dtype=np.float64)
            frame = metadata.copy()
            frame.insert(0, "state", state)
            frame.insert(0, "mask_id", mask_id)
            frame.insert(0, "output_type", output_name)
            frame["resolution_bp"] = int(track_data.resolution)
            frame["mask_bins"] = int(mask.sum())
            frame["mask_effective_bp"] = int(mask.sum() * track_data.resolution)
            frame["signal_sum"] = signal_sum
            frame["log2_signal"] = np.log2(signal_sum + EPS)
            frames.append(frame)
    if not frames:
        raise RuntimeError("AlphaGenome no devolvió los outputs solicitados")
    return pd.concat(frames, ignore_index=True, sort=False)


def predict_with_retry(variant):
    last_error = None
    for attempt in range(API_RETRIES):
        try:
            return dna_model.predict_variant(
                interval=FIXED_INTERVAL,
                variant=variant,
                organism=dna_client.Organism.HOMO_SAPIENS,
                requested_outputs=[spec[0] for spec in OUTPUT_SPECS.values()],
                ontology_terms=requested_terms,
            )
        except Exception as exc:
            last_error = exc
            if attempt + 1 == API_RETRIES:
                raise
            wait = RETRY_BASE_SECONDS * (2 ** attempt)
            print(f"  Reintento {attempt + 1}/{API_RETRIES - 1} en {wait}s: {exc}")
            time.sleep(wait)
    raise last_error


def compare_reference_summaries(base, other, label):
    keys = ["output_type", "mask_id", "track_key"]
    merged = base[keys + ["log2_signal"]].merge(
        other[keys + ["log2_signal"]],
        on=keys, how="outer", suffixes=("_base", "_other"), indicator=True,
    )
    rows = []
    for (output_type, mask_id), group in merged.groupby(
        ["output_type", "mask_id"], dropna=False
    ):
        complete = group["_merge"].eq("both")
        diffs = (
            group.loc[complete, "log2_signal_base"] -
            group.loc[complete, "log2_signal_other"]
        ).abs()
        rows.append({
            "comparison": label,
            "output_type": output_type,
            "mask_id": mask_id,
            "n_base": int(group["_merge"].isin(["both", "left_only"]).sum()),
            "n_other": int(group["_merge"].isin(["both", "right_only"]).sum()),
            "n_shared": int(complete.sum()),
            "n_missing_either": int((~complete).sum()),
            "max_abs_log2sum_diff": float(diffs.max()) if len(diffs) else np.nan,
        })
    return pd.DataFrame(rows)


def run_pair(row):
    pair_id = str(row["Somatic_Event_ID"])
    germline, somatic, combined = pair_variants(row)

    prediction_g = predict_with_retry(germline)
    wt_g = summarise_output(prediction_g.reference, "WT_GERMLINE_CALL", row)
    state_g = summarise_output(prediction_g.alternate, "GERMLINE", row)
    del prediction_g
    gc.collect()

    prediction_s = predict_with_retry(somatic)
    wt_s = summarise_output(prediction_s.reference, "WT_SOMATIC_CALL", row)
    state_s = summarise_output(prediction_s.alternate, "SOMATIC", row)
    del prediction_s
    gc.collect()

    prediction_c = predict_with_retry(combined)
    wt_c = summarise_output(prediction_c.reference, "WT_CIS_CALL", row)
    state_c = summarise_output(prediction_c.alternate, "COMBINED_CIS", row)
    del prediction_c
    gc.collect()

    qc = pd.concat([
        compare_reference_summaries(wt_g, wt_s, "WT_G_vs_WT_S"),
        compare_reference_summaries(wt_g, wt_c, "WT_G_vs_WT_CIS"),
    ], ignore_index=True)
    qc.insert(0, "pair_id", pair_id)
    contains_indel = str(row.get("contains_indel", False)).strip().lower() in {"true", "1", "yes"}
    qc["contains_indel"] = contains_indel
    qc["reference_identity_expected"] = not contains_indel
    qc["track_alignment_pass"] = qc["n_missing_either"].eq(0)
    qc["reference_identity_pass"] = qc[
        "max_abs_log2sum_diff"
    ].fillna(np.inf).le(REF_QC_TOL)
    qc["qc_pass"] = (
        qc["track_alignment_pass"] &
        (~qc["reference_identity_expected"] | qc["reference_identity_pass"])
    )
    qc.to_csv(QC_DIR / f"{pair_id}_reference_consistency.csv", index=False)

    states = pd.concat(
        [wt_g, state_g, wt_s, state_s, wt_c, state_c],
        ignore_index=True, sort=False,
    )
    states.insert(0, "pair_id", pair_id)
    states.to_parquet(STATE_DIR / f"{pair_id}_state_summaries.parquet", index=False)

    keys = ["output_type", "mask_id", "track_key"]
    wide = states.pivot_table(
        index=keys, columns="state", values="log2_signal", aggfunc="first"
    ).reset_index()
    metadata_cols = [
        col for col in states.columns
        if col not in {"pair_id", "state", "signal_sum", "log2_signal"}
        and col not in keys
    ]
    metadata_one = states[
        states["state"].eq("WT_GERMLINE_CALL")
    ][keys + metadata_cols].drop_duplicates(keys)
    metrics = metadata_one.merge(wide, on=keys, how="outer", validate="one_to_one")

    paired_states = [
        "WT_GERMLINE_CALL", "GERMLINE",
        "WT_SOMATIC_CALL", "SOMATIC",
        "WT_CIS_CALL", "COMBINED_CIS",
    ]
    for state in paired_states:
        if state not in metrics.columns:
            metrics[state] = np.nan
    metrics["complete_paired_effects"] = metrics[paired_states].notna().all(axis=1)
    # Alias conservado para que las celdas de resumen sigan siendo compatibles.
    metrics["complete_four_state"] = metrics["complete_paired_effects"]
    metrics["germline_effect"] = (
        metrics["GERMLINE"] - metrics["WT_GERMLINE_CALL"]
    )
    metrics["somatic_effect"] = (
        metrics["SOMATIC"] - metrics["WT_SOMATIC_CALL"]
    )
    metrics["combined_cis_effect"] = (
        metrics["COMBINED_CIS"] - metrics["WT_CIS_CALL"]
    )
    metrics["abs_germline_effect"] = metrics["germline_effect"].abs()
    metrics["abs_somatic_effect"] = metrics["somatic_effect"].abs()
    metrics["abs_combined_cis_effect"] = metrics["combined_cis_effect"].abs()
    metrics["absolute_improvement_to_WT"] = (
        metrics["abs_germline_effect"] - metrics["abs_combined_cis_effect"]
    )
    eligible_denominator = metrics["abs_germline_effect"] >= MIN_ABS_G_EFFECT
    metrics["rescue_to_WT_ratio"] = np.where(
        eligible_denominator,
        1 - metrics["abs_combined_cis_effect"] / metrics["abs_germline_effect"],
        np.nan,
    )
    metrics["closer_to_WT"] = np.where(
        eligible_denominator,
        metrics["abs_combined_cis_effect"] < metrics["abs_germline_effect"],
        np.nan,
    )
    metrics["interaction_log2sum"] = (
        metrics["combined_cis_effect"] -
        (metrics["germline_effect"] + metrics["somatic_effect"])
    )

    insert_values = [
        ("pair_id", pair_id),
        ("patient", row["patient"]),
        ("background", row["background"]),
        ("germline_U1", row["germline_U1"]),
        ("germline_protein", row["germline_protein"]),
        ("somatic_ID", row["somatic_ID"]),
        ("somatic_protein", row["somatic_protein"]),
        ("somatic_variant_class", row.get("somatic_variant_class", "")),
        ("pair_span_bp", row.get("pair_span_bp", np.nan)),
        ("contains_indel", str(row.get("contains_indel", False)).strip().lower() in {"true", "1", "yes"}),
    ]
    for position, (column, value) in enumerate(insert_values):
        metrics.insert(position, column, value)
    metrics.to_parquet(PAIR_DIR / f"{pair_id}_metrics.parquet", index=False)

    return metrics, qc, str(combined)

In [ ]:
# 7) Preflight: comprobar GRCh38 y construir los 54 haplotipos antes de gastar API
preflight_rows = []
for index, row in manifest.iterrows():
    pair_id = row["Somatic_Event_ID"]
    try:
        germline, somatic, combined = pair_variants(row)
        preflight_rows.append({
            "pair_id": pair_id,
            "status": "OK",
            "combined_position": combined.position,
            "combined_ref_length": len(combined.reference_bases),
            "combined_alt_length": len(combined.alternate_bases),
            "contains_indel": str(row["contains_indel"]).strip().lower() in {"true", "1", "yes"},
            "error": "",
        })
    except Exception as exc:
        preflight_rows.append({
            "pair_id": pair_id, "status": "FAILED",
            "combined_position": np.nan, "combined_ref_length": np.nan,
            "combined_alt_length": np.nan,
            "contains_indel": str(row["contains_indel"]).strip().lower() in {"true", "1", "yes"},
            "error": repr(exc),
        })

preflight = pd.DataFrame(preflight_rows)
preflight.to_csv(QC_DIR / "haplotype_preflight.csv", index=False)
display(preflight)
if not preflight["status"].eq("OK").all():
    raise RuntimeError("Preflight fallido; corrige REF/coordenadas antes de llamar a AlphaGenome")
print("✅ Todos los haplotipos validados contra GRCh38")

In [ ]:
# 8) Ejecutar. Hay checkpoint por par; se puede interrumpir y volver a lanzar.
rows_to_run = manifest if MAX_PAIRS is None else manifest.head(int(MAX_PAIRS))
run_audit = []

for number, (_, row) in enumerate(rows_to_run.iterrows(), start=1):
    pair_id = str(row["Somatic_Event_ID"])
    metric_path = PAIR_DIR / f"{pair_id}_metrics.parquet"
    t0 = time.time()
    qc_path = QC_DIR / f"{pair_id}_reference_consistency.csv"
    if (metric_path.exists() and qc_path.exists() and not FORCE_RERUN
            and pd.read_csv(qc_path)['qc_pass'].astype(str).str.lower().eq('true').all()):
        run_audit.append({
            "pair_id": pair_id, "status": "SKIPPED_CHECKPOINT",
            "seconds": 0.0, "rows": len(pd.read_parquet(metric_path)), "error": "",
        })
        print(f"[{number:02d}/{len(rows_to_run):02d}] ↪ {pair_id} (checkpoint)")
        continue
    try:
        metrics, qc, combined_text = run_pair(row)
        if bool(qc["qc_pass"].all()):
            contains_indel = str(row.get("contains_indel", False)).strip().lower() in {"true", "1", "yes"}
            status = "OK_PAIRED_REFERENCE_INDEL" if contains_indel else "OK"
        else:
            status = "QC_FAILED"
        run_audit.append({
            "pair_id": pair_id, "status": status,
            "seconds": round(time.time() - t0, 2), "rows": len(metrics),
            "combined_variant": combined_text, "error": "",
        })
        icon = "✅" if status in {"OK", "OK_PAIRED_REFERENCE_INDEL"} else "⚠️"
        print(
            f"[{number:02d}/{len(rows_to_run):02d}] {icon} {pair_id}: "
            f"{row['germline_protein']} + {row['somatic_protein']} "
            f"({time.time() - t0:.1f}s)"
        )
    except Exception as exc:
        run_audit.append({
            "pair_id": pair_id, "status": "FAILED",
            "seconds": round(time.time() - t0, 2), "rows": 0,
            "combined_variant": "", "error": repr(exc),
        })
        print(f"[{number:02d}/{len(rows_to_run):02d}] ❌ {pair_id}: {exc}")
    finally:
        gc.collect()

run_audit_df = pd.DataFrame(run_audit)
run_audit_df.to_csv(OUT / "run_audit.csv", index=False)
display(run_audit_df)

ok = run_audit_df["status"].isin([
    "OK", "OK_PAIRED_REFERENCE_INDEL", "SKIPPED_CHECKPOINT"
]).sum()
print(f"Completados/checkpoint: {ok}/{len(rows_to_run)}")

In [ ]:
from google.colab import files
import shutil
assert set(manifest.Somatic_Event_ID)=={'CTRL031', 'CTRL_SGR056_2', 'SGR013', 'CTRL162', 'CTRL021', 'CTRL003', 'CTRL094', 'CTRL051', 'CTRL076', 'CTRL071', 'CTRL023', 'CTRL005', 'CTRL098', 'CTRL062', 'CTRL015', 'CTRL_SGR057_3', 'CTRL096', 'CTRL032', 'CTRL026', 'CTRL163', 'CTRL156', 'SGR052', 'CTRL029', 'SGR054', 'SGR014', 'CTRL013', 'CTRL041', 'CTRL047', 'CTRL164', 'SGR006', 'CTRL102', 'CTRL018', 'CTRL150', 'CTRL126', 'CTRL078', 'CTRL_SGR056_1', 'CTRL139', 'CTRL099', 'CTRL120', 'CTRL100', 'SGR004', 'CTRL019', 'CTRL160', 'CTRL042', 'CTRL_SGR057_4', 'CTRL_SGR056_3', 'CTRL039', 'CTRL147', 'SGR030', 'CTRL144', 'CTRL053', 'SGR017', 'CTRL008', 'CTRL152', 'CTRL074', 'SGR035', 'CTRL065', 'CTRL103', 'CTRL082', 'SGR026', 'SGR009', 'CTRL109', 'SGR053', 'CTRL155', 'CTRL061', 'SGR045', 'CTRL104', 'SGR043', 'CTRL007', 'SGR042', 'CTRL055', 'CTRL161', 'SGR001', 'CTRL131', 'SGR050', 'CTRL133', 'CTRL_SGR057_2', 'CTRL124', 'CTRL022', 'CTRL083', 'SGR019', 'CTRL148', 'CTRL136', 'SGR034', 'CTRL091', 'CTRL072', 'SGR029', 'CTRL115', 'CTRL134', 'SGR003', 'SGR007', 'SGR024', 'CTRL119', 'CTRL080', 'CTRL116', 'CTRL122', 'CTRL_SGR056_4', 'CTRL064', 'CTRL010', 'CTRL089', 'CTRL130', 'SGR046', 'CTRL101', 'SGR036', 'CTRL002', 'CTRL138', 'SGR020', 'CTRL154', 'CTRL040', 'CTRL014', 'CTRL069', 'CTRL017', 'CTRL_SGR056_5', 'CTRL033', 'CTRL024', 'CTRL143', 'CTRL067', 'CTRL025', 'CTRL027', 'CTRL111', 'CTRL165', 'SGR011', 'SGR047', 'CTRL066', 'CTRL037', 'CTRL137', 'CTRL085', 'CTRL077', 'CTRL063', 'CTRL075', 'CTRL050', 'CTRL020', 'CTRL132', 'CTRL108', 'SGR015', 'SGR002', 'SGR056', 'CTRL153', 'CTRL097', 'CTRL128', 'CTRL113', 'CTRL118', 'CTRL035', 'CTRL059', 'SGR018', 'CTRL046', 'CTRL049', 'CTRL038', 'CTRL086', 'CTRL081', 'CTRL012', 'SGR039', 'SGR055', 'SGR012', 'CTRL151', 'CTRL057', 'SGR022', 'SGR031', 'CTRL052', 'CTRL095', 'SGR049', 'SGR005', 'CTRL011', 'CTRL145', 'CTRL030', 'CTRL127', 'SGR010', 'CTRL106', 'CTRL056', 'SGR021', 'CTRL004', 'SGR038', 'SGR057', 'CTRL149', 'SGR040', 'SGR051', 'CTRL112', 'CTRL_SGR057_5', 'CTRL045', 'SGR048', 'CTRL140', 'CTRL073', 'CTRL_SGR057_1', 'SGR028', 'CTRL060', 'CTRL090', 'CTRL159', 'SGR032', 'SGR041', 'SGR027', 'CTRL054', 'CTRL006', 'CTRL117', 'CTRL079', 'CTRL001', 'CTRL028', 'CTRL125', 'CTRL107', 'CTRL068', 'CTRL048', 'CTRL146', 'CTRL114', 'CTRL044', 'CTRL087', 'CTRL157', 'CTRL135', 'CTRL141', 'CTRL084', 'CTRL009', 'CTRL158', 'CTRL070', 'CTRL123', 'CTRL121', 'CTRL110', 'SGR023', 'SGR033', 'SGR037', 'CTRL142', 'CTRL016', 'CTRL105', 'CTRL092', 'CTRL058', 'CTRL043', 'CTRL093', 'CTRL129', 'SGR008', 'CTRL088', 'CTRL036', 'CTRL034'}
audit = pd.read_csv(OUT/'run_audit.csv')
assert set(audit.pair_id)==set(manifest.Somatic_Event_ID)
assert audit.status.isin(['OK','OK_PAIRED_REFERENCE_INDEL','SKIPPED_CHECKPOINT']).all(), audit
import io
assignments=pd.read_csv(io.StringIO('observed_pair_id,observed_background,germline_variant_id,observed_somatic_variant_id,control_somatic_variant_id,control_source_pair_id,control_pair_id,match_tier\nSGR001,I891T,chr7:93133300:A>G,chr7:93133670:T>A,chr7:93133634:G>A,SGR021,CTRL143,0\nSGR001,I891T,chr7:93133300:A>G,chr7:93133670:T>A,chr7:93133587:G>T,SGR052,CTRL142,0\nSGR001,I891T,chr7:93133300:A>G,chr7:93133670:T>A,chr7:93133490:G>A,SGR036,CTRL141,0\nSGR001,I891T,chr7:93133300:A>G,chr7:93133670:T>A,chr7:93134207:G>A,SGR017,CTRL145,1\nSGR001,I891T,chr7:93133300:A>G,chr7:93133670:T>A,chr7:93133645:T>TC,SGR015,CTRL144,3\nSGR002,R986C,chr7:93133016:G>A,chr7:93133015:C>A,chr7:93132949:T>A,SGR030,CTRL115,4\nSGR002,R986C,chr7:93133016:G>A,chr7:93133015:C>A,chr7:93132814:G>A,SGR025,CTRL114,4\nSGR002,R986C,chr7:93133016:G>A,chr7:93133015:C>A,chr7:93133273:T>C,SGR049,CTRL116,4\nSGR002,R986C,chr7:93133016:G>A,chr7:93133015:C>A,chr7:93132698:A>C,SGR026,CTRL113,4\nSGR002,R986C,chr7:93133016:G>A,chr7:93133015:C>A,chr7:93132608:T>A,SGR007,CTRL112,4\nSGR003,R986C,chr7:93133016:G>A,chr7:93134763:TATGAG>T,chr7:93134746:AG>A,SGR018,CTRL123,0\nSGR003,R986C,chr7:93133016:G>A,chr7:93134763:TATGAG>T,chr7:93134889:TA>T,SGR012,CTRL124,0\nSGR003,R986C,chr7:93133016:G>A,chr7:93134763:TATGAG>T,chr7:93134586:ACTTT>A,SGR029,CTRL122,0\nSGR003,R986C,chr7:93133016:G>A,chr7:93134763:TATGAG>T,chr7:93134986:T>TG,SGR055,CTRL125,0\nSGR003,R986C,chr7:93133016:G>A,chr7:93134763:TATGAG>T,chr7:93134149:C>CTG,SGR028,CTRL121,0\nSGR004,R986C,chr7:93133016:G>A,chr7:93132635:C>T,chr7:93132608:T>A,SGR007,CTRL112,0\nSGR004,R986C,chr7:93133016:G>A,chr7:93132635:C>T,chr7:93132698:A>C,SGR026,CTRL113,0\nSGR004,R986C,chr7:93133016:G>A,chr7:93132635:C>T,chr7:93132461:C>T,SGR027,CTRL111,0\nSGR004,R986C,chr7:93133016:G>A,chr7:93132635:C>T,chr7:93132814:G>A,SGR025,CTRL114,0\nSGR004,R986C,chr7:93133016:G>A,chr7:93132635:C>T,chr7:93132052:A>G,SGR046,CTRL110,0\nSGR005,H880Q,chr7:93133332:G>T,chr7:93132016:TCTTTA>T,chr7:93131747:G>GT,SGR051,CTRL151,0\nSGR005,H880Q,chr7:93133332:G>T,chr7:93132016:TCTTTA>T,chr7:93132321:G>GA,SGR053,CTRL152,0\nSGR005,H880Q,chr7:93133332:G>T,chr7:93132016:TCTTTA>T,chr7:93132410:G>A,SGR019,CTRL153,3\nSGR005,H880Q,chr7:93133332:G>T,chr7:93132016:TCTTTA>T,chr7:93132425:G>A,SGR006,CTRL154,3\nSGR005,H880Q,chr7:93133332:G>T,chr7:93132016:TCTTTA>T,chr7:93132432:C>T,SGR031,CTRL155,3\nSGR006,R1281K,chr7:93132130:C>T,chr7:93132425:G>A,chr7:93132432:C>T,SGR031,CTRL057,0\nSGR006,R1281K,chr7:93132130:C>T,chr7:93132425:G>A,chr7:93132743:G>A,SGR040,CTRL062,0\nSGR006,R1281K,chr7:93132130:C>T,chr7:93132425:G>A,chr7:93132794:C>A,SGR037,CTRL063,0\nSGR006,R1281K,chr7:93132130:C>T,chr7:93132425:G>A,chr7:93133490:G>A,SGR036,CTRL066,0\nSGR006,R1281K,chr7:93132130:C>T,chr7:93132425:G>A,chr7:93133587:G>T,SGR052,CTRL067,0\nSGR007,R1281K,chr7:93132130:C>T,chr7:93132608:T>A,chr7:93132635:C>T,SGR004,CTRL058,0\nSGR007,R1281K,chr7:93132130:C>T,chr7:93132608:T>A,chr7:93132689:T>C,SGR045,CTRL059,0\nSGR007,R1281K,chr7:93132130:C>T,chr7:93132608:T>A,chr7:93132696:G>C,SGR020,CTRL060,0\nSGR007,R1281K,chr7:93132130:C>T,chr7:93132608:T>A,chr7:93132742:C>T,SGR034,CTRL061,0\nSGR007,R1281K,chr7:93132130:C>T,chr7:93132608:T>A,chr7:93132949:T>A,SGR030,CTRL064,0\nSGR008,R1281K,chr7:93132130:C>T,chr7:93132601:CCTAGTG>C,chr7:93132635:C>T,SGR004,CTRL058,3\nSGR008,R1281K,chr7:93132130:C>T,chr7:93132601:CCTAGTG>C,chr7:93132689:T>C,SGR045,CTRL059,3\nSGR008,R1281K,chr7:93132130:C>T,chr7:93132601:CCTAGTG>C,chr7:93132696:G>C,SGR020,CTRL060,3\nSGR008,R1281K,chr7:93132130:C>T,chr7:93132601:CCTAGTG>C,chr7:93132742:C>T,SGR034,CTRL061,3\nSGR008,R1281K,chr7:93132130:C>T,chr7:93132601:CCTAGTG>C,chr7:93132949:T>A,SGR030,CTRL064,3\nSGR009,R1281K,chr7:93132130:C>T,chr7:93131783:A>G,chr7:93131401:C>T,SGR043,CTRL054,0\nSGR009,R1281K,chr7:93132130:C>T,chr7:93131783:A>G,chr7:93132052:A>G,SGR046,CTRL055,0\nSGR009,R1281K,chr7:93132130:C>T,chr7:93131783:A>G,chr7:93132635:C>T,SGR004,CTRL058,4\nSGR009,R1281K,chr7:93132130:C>T,chr7:93131783:A>G,chr7:93132689:T>C,SGR045,CTRL059,4\nSGR009,R1281K,chr7:93132130:C>T,chr7:93131783:A>G,chr7:93132696:G>C,SGR020,CTRL060,4\nSGR010,R1281K,chr7:93132130:C>T,chr7:93133015:C>A,chr7:93132949:T>A,SGR030,CTRL064,1\nSGR010,R1281K,chr7:93132130:C>T,chr7:93133015:C>A,chr7:93133273:T>C,SGR049,CTRL065,1\nSGR010,R1281K,chr7:93132130:C>T,chr7:93133015:C>A,chr7:93132742:C>T,SGR034,CTRL061,1\nSGR010,R1281K,chr7:93132130:C>T,chr7:93133015:C>A,chr7:93132696:G>C,SGR020,CTRL060,1\nSGR010,R1281K,chr7:93132130:C>T,chr7:93133015:C>A,chr7:93132689:T>C,SGR045,CTRL059,1\nSGR011,R1281K,chr7:93132130:C>T,chr7:93132513:TAGG>T,chr7:93132635:C>T,SGR004,CTRL058,3\nSGR011,R1281K,chr7:93132130:C>T,chr7:93132513:TAGG>T,chr7:93132689:T>C,SGR045,CTRL059,3\nSGR011,R1281K,chr7:93132130:C>T,chr7:93132513:TAGG>T,chr7:93132696:G>C,SGR020,CTRL060,3\nSGR011,R1281K,chr7:93132130:C>T,chr7:93132513:TAGG>T,chr7:93132742:C>T,SGR034,CTRL061,3\nSGR011,R1281K,chr7:93132130:C>T,chr7:93132513:TAGG>T,chr7:93132949:T>A,SGR030,CTRL064,3\nSGR012,R1281K,chr7:93132130:C>T,chr7:93134889:TA>T,chr7:93134986:T>TG,SGR055,CTRL076,0\nSGR012,R1281K,chr7:93132130:C>T,chr7:93134889:TA>T,chr7:93134763:TATGAG>T,SGR003,CTRL075,0\nSGR012,R1281K,chr7:93132130:C>T,chr7:93134889:TA>T,chr7:93134746:AG>A,SGR018,CTRL074,0\nSGR012,R1281K,chr7:93132130:C>T,chr7:93134889:TA>T,chr7:93134586:ACTTT>A,SGR029,CTRL072,0\nSGR012,R1281K,chr7:93132130:C>T,chr7:93134889:TA>T,chr7:93134149:C>CTG,SGR028,CTRL071,0\nSGR013,R1281K,chr7:93132130:C>T,chr7:93135916:A>T,chr7:93135289:C>T,SGR048,CTRL077,0\nSGR013,R1281K,chr7:93132130:C>T,chr7:93135916:A>T,chr7:93134612:C>T,SGR042,CTRL073,0\nSGR013,R1281K,chr7:93132130:C>T,chr7:93135916:A>T,chr7:93133887:C>G,SGR032,CTRL070,0\nSGR013,R1281K,chr7:93132130:C>T,chr7:93135916:A>T,chr7:93133273:T>C,SGR049,CTRL065,0\nSGR013,R1281K,chr7:93132130:C>T,chr7:93135916:A>T,chr7:93132949:T>A,SGR030,CTRL064,0\nSGR014,R1281K,chr7:93132130:C>T,chr7:93134288:C>T,chr7:93134612:C>T,SGR042,CTRL073,0\nSGR014,R1281K,chr7:93132130:C>T,chr7:93134288:C>T,chr7:93133887:C>G,SGR032,CTRL070,0\nSGR014,R1281K,chr7:93132130:C>T,chr7:93134288:C>T,chr7:93135289:C>T,SGR048,CTRL077,0\nSGR014,R1281K,chr7:93132130:C>T,chr7:93134288:C>T,chr7:93133273:T>C,SGR049,CTRL065,0\nSGR014,R1281K,chr7:93132130:C>T,chr7:93134288:C>T,chr7:93132949:T>A,SGR030,CTRL064,0\nSGR015,R1281K,chr7:93132130:C>T,chr7:93133645:T>TC,chr7:93134149:C>CTG,SGR028,CTRL071,0\nSGR015,R1281K,chr7:93132130:C>T,chr7:93133645:T>TC,chr7:93134586:ACTTT>A,SGR029,CTRL072,0\nSGR015,R1281K,chr7:93132130:C>T,chr7:93133645:T>TC,chr7:93134746:AG>A,SGR018,CTRL074,0\nSGR015,R1281K,chr7:93132130:C>T,chr7:93133645:T>TC,chr7:93134763:TATGAG>T,SGR003,CTRL075,0\nSGR015,R1281K,chr7:93132130:C>T,chr7:93133645:T>TC,chr7:93134986:T>TG,SGR055,CTRL076,0\nSGR017,R1281K,chr7:93132130:C>T,chr7:93134207:G>A,chr7:93132410:G>A,SGR019,CTRL056,0\nSGR017,R1281K,chr7:93132130:C>T,chr7:93134207:G>A,chr7:93133670:T>A,SGR001,CTRL069,1\nSGR017,R1281K,chr7:93132130:C>T,chr7:93134207:G>A,chr7:93133634:G>A,SGR021,CTRL068,1\nSGR017,R1281K,chr7:93132130:C>T,chr7:93134207:G>A,chr7:93133587:G>T,SGR052,CTRL067,1\nSGR017,R1281K,chr7:93132130:C>T,chr7:93134207:G>A,chr7:93133490:G>A,SGR036,CTRL066,1\nSGR018,V1512A,chr7:93131437:A>G,chr7:93134746:AG>A,chr7:93134763:TATGAG>T,SGR003,CTRL013,0\nSGR018,V1512A,chr7:93131437:A>G,chr7:93134746:AG>A,chr7:93134889:TA>T,SGR012,CTRL014,0\nSGR018,V1512A,chr7:93131437:A>G,chr7:93134746:AG>A,chr7:93134586:ACTTT>A,SGR029,CTRL012,0\nSGR018,V1512A,chr7:93131437:A>G,chr7:93134746:AG>A,chr7:93134986:T>TG,SGR055,CTRL015,0\nSGR018,V1512A,chr7:93131437:A>G,chr7:93134746:AG>A,chr7:93134149:C>CTG,SGR028,CTRL011,0\nSGR019,V1512M,chr7:93131438:C>T,chr7:93132410:G>A,chr7:93134207:G>A,SGR017,CTRL020,0\nSGR019,V1512M,chr7:93131438:C>T,chr7:93132410:G>A,chr7:93132425:G>A,SGR006,CTRL016,1\nSGR019,V1512M,chr7:93131438:C>T,chr7:93132410:G>A,chr7:93132432:C>T,SGR031,CTRL017,1\nSGR019,V1512M,chr7:93131438:C>T,chr7:93132410:G>A,chr7:93132743:G>A,SGR040,CTRL018,1\nSGR019,V1512M,chr7:93131438:C>T,chr7:93132410:G>A,chr7:93132794:C>A,SGR037,CTRL019,1\nSGR020,R986C,chr7:93133016:G>A,chr7:93132696:G>C,chr7:93132698:A>C,SGR026,CTRL113,0\nSGR020,R986C,chr7:93133016:G>A,chr7:93132696:G>C,chr7:93132608:T>A,SGR007,CTRL112,0\nSGR020,R986C,chr7:93133016:G>A,chr7:93132696:G>C,chr7:93132814:G>A,SGR025,CTRL114,0\nSGR020,R986C,chr7:93133016:G>A,chr7:93132696:G>C,chr7:93132461:C>T,SGR027,CTRL111,0\nSGR020,R986C,chr7:93133016:G>A,chr7:93132696:G>C,chr7:93132052:A>G,SGR046,CTRL110,0\nSGR021,V1512A,chr7:93131437:A>G,chr7:93133634:G>A,chr7:93133670:T>A,SGR001,CTRL010,0\nSGR021,V1512A,chr7:93131437:A>G,chr7:93133634:G>A,chr7:93133587:G>T,SGR052,CTRL009,0\nSGR021,V1512A,chr7:93131437:A>G,chr7:93133634:G>A,chr7:93133490:G>A,SGR036,CTRL008,0\nSGR021,V1512A,chr7:93131437:A>G,chr7:93133634:G>A,chr7:93132794:C>A,SGR037,CTRL007,0\nSGR021,V1512A,chr7:93131437:A>G,chr7:93133634:G>A,chr7:93132743:G>A,SGR040,CTRL006,0\nSGR022,R1281K,chr7:93132130:C>T,chr7:93135764:G>A,chr7:93135289:C>T,SGR048,CTRL077,0\nSGR022,R1281K,chr7:93132130:C>T,chr7:93135764:G>A,chr7:93134612:C>T,SGR042,CTRL073,0\nSGR022,R1281K,chr7:93132130:C>T,chr7:93135764:G>A,chr7:93133887:C>G,SGR032,CTRL070,0\nSGR022,R1281K,chr7:93132130:C>T,chr7:93135764:G>A,chr7:93133273:T>C,SGR049,CTRL065,0\nSGR022,R1281K,chr7:93132130:C>T,chr7:93135764:G>A,chr7:93132949:T>A,SGR030,CTRL064,0\nSGR023,R1281K,chr7:93132130:C>T,chr7:93135134:C>T,chr7:93135289:C>T,SGR048,CTRL077,0\nSGR023,R1281K,chr7:93132130:C>T,chr7:93135134:C>T,chr7:93134612:C>T,SGR042,CTRL073,0\nSGR023,R1281K,chr7:93132130:C>T,chr7:93135134:C>T,chr7:93133887:C>G,SGR032,CTRL070,0\nSGR023,R1281K,chr7:93132130:C>T,chr7:93135134:C>T,chr7:93133273:T>C,SGR049,CTRL065,0\nSGR023,R1281K,chr7:93132130:C>T,chr7:93135134:C>T,chr7:93132949:T>A,SGR030,CTRL064,0\nSGR024,R1281K,chr7:93132130:C>T,chr7:93134896:C>T,chr7:93134612:C>T,SGR042,CTRL073,0\nSGR024,R1281K,chr7:93132130:C>T,chr7:93134896:C>T,chr7:93135289:C>T,SGR048,CTRL077,0\nSGR024,R1281K,chr7:93132130:C>T,chr7:93134896:C>T,chr7:93133887:C>G,SGR032,CTRL070,0\nSGR024,R1281K,chr7:93132130:C>T,chr7:93134896:C>T,chr7:93133273:T>C,SGR049,CTRL065,0\nSGR024,R1281K,chr7:93132130:C>T,chr7:93134896:C>T,chr7:93132949:T>A,SGR030,CTRL064,0\nSGR026,R1281K,chr7:93132130:C>T,chr7:93132698:A>C,chr7:93132696:G>C,SGR020,CTRL060,0\nSGR026,R1281K,chr7:93132130:C>T,chr7:93132698:A>C,chr7:93132689:T>C,SGR045,CTRL059,0\nSGR026,R1281K,chr7:93132130:C>T,chr7:93132698:A>C,chr7:93132742:C>T,SGR034,CTRL061,0\nSGR026,R1281K,chr7:93132130:C>T,chr7:93132698:A>C,chr7:93132635:C>T,SGR004,CTRL058,0\nSGR026,R1281K,chr7:93132130:C>T,chr7:93132698:A>C,chr7:93132949:T>A,SGR030,CTRL064,0\nSGR027,R1281K,chr7:93132130:C>T,chr7:93132461:C>T,chr7:93132635:C>T,SGR004,CTRL058,0\nSGR027,R1281K,chr7:93132130:C>T,chr7:93132461:C>T,chr7:93132689:T>C,SGR045,CTRL059,0\nSGR027,R1281K,chr7:93132130:C>T,chr7:93132461:C>T,chr7:93132696:G>C,SGR020,CTRL060,0\nSGR027,R1281K,chr7:93132130:C>T,chr7:93132461:C>T,chr7:93132742:C>T,SGR034,CTRL061,0\nSGR027,R1281K,chr7:93132130:C>T,chr7:93132461:C>T,chr7:93132949:T>A,SGR030,CTRL064,0\nSGR028,S1473N,chr7:93131554:C>T,chr7:93134149:C>CTG,chr7:93134746:AG>A,SGR018,CTRL045,0\nSGR028,S1473N,chr7:93131554:C>T,chr7:93134149:C>CTG,chr7:93134763:TATGAG>T,SGR003,CTRL046,0\nSGR028,S1473N,chr7:93131554:C>T,chr7:93134149:C>CTG,chr7:93133645:T>TC,SGR015,CTRL044,0\nSGR028,S1473N,chr7:93131554:C>T,chr7:93134149:C>CTG,chr7:93134889:TA>T,SGR012,CTRL047,0\nSGR028,S1473N,chr7:93131554:C>T,chr7:93134149:C>CTG,chr7:93134986:T>TG,SGR055,CTRL048,0\nSGR029,S1473N,chr7:93131554:C>T,chr7:93134586:ACTTT>A,chr7:93134746:AG>A,SGR018,CTRL045,0\nSGR029,S1473N,chr7:93131554:C>T,chr7:93134586:ACTTT>A,chr7:93134763:TATGAG>T,SGR003,CTRL046,0\nSGR029,S1473N,chr7:93131554:C>T,chr7:93134586:ACTTT>A,chr7:93134889:TA>T,SGR012,CTRL047,0\nSGR029,S1473N,chr7:93131554:C>T,chr7:93134586:ACTTT>A,chr7:93134986:T>TG,SGR055,CTRL048,0\nSGR029,S1473N,chr7:93131554:C>T,chr7:93134586:ACTTT>A,chr7:93133645:T>TC,SGR015,CTRL044,0\nSGR030,S1509R,chr7:93131447:T>G,chr7:93132949:T>A,chr7:93132814:G>A,SGR025,CTRL036,0\nSGR030,S1509R,chr7:93131447:T>G,chr7:93132949:T>A,chr7:93132742:C>T,SGR034,CTRL033,0\nSGR030,S1509R,chr7:93131447:T>G,chr7:93132949:T>A,chr7:93132698:A>C,SGR026,CTRL032,0\nSGR030,S1509R,chr7:93131447:T>G,chr7:93132949:T>A,chr7:93132696:G>C,SGR020,CTRL031,0\nSGR030,S1509R,chr7:93131447:T>G,chr7:93132949:T>A,chr7:93132689:T>C,SGR045,CTRL030,0\nSGR031,S1509R,chr7:93131447:T>G,chr7:93132432:C>T,chr7:93132425:G>A,SGR006,CTRL029,0\nSGR031,S1509R,chr7:93131447:T>G,chr7:93132432:C>T,chr7:93132743:G>A,SGR040,CTRL034,0\nSGR031,S1509R,chr7:93131447:T>G,chr7:93132432:C>T,chr7:93132794:C>A,SGR037,CTRL035,0\nSGR031,S1509R,chr7:93131447:T>G,chr7:93132432:C>T,chr7:93133490:G>A,SGR036,CTRL037,0\nSGR031,S1509R,chr7:93131447:T>G,chr7:93132432:C>T,chr7:93133587:G>T,SGR052,CTRL038,0\nSGR032,R986H,chr7:93133015:C>T,chr7:93133887:C>G,chr7:93134288:C>T,SGR014,CTRL104,0\nSGR032,R986H,chr7:93133015:C>T,chr7:93133887:C>G,chr7:93134612:C>T,SGR042,CTRL105,0\nSGR032,R986H,chr7:93133015:C>T,chr7:93133887:C>G,chr7:93133445:G>A,SGR016,CTRL099,0\nSGR032,R986H,chr7:93133015:C>T,chr7:93133887:C>G,chr7:93134896:C>T,SGR024,CTRL106,0\nSGR032,R986H,chr7:93133015:C>T,chr7:93133887:C>G,chr7:93135134:C>T,SGR023,CTRL107,0\nSGR033,P1349L,chr7:93131926:G>A,chr7:93135203:T>TA,chr7:93133402:AGTGC>A,SGR054,CTRL049,0\nSGR033,P1349L,chr7:93131926:G>A,chr7:93135203:T>TA,chr7:93134986:T>TG,SGR055,CTRL053,2\nSGR033,P1349L,chr7:93131926:G>A,chr7:93135203:T>TA,chr7:93134889:TA>T,SGR012,CTRL052,2\nSGR033,P1349L,chr7:93131926:G>A,chr7:93135203:T>TA,chr7:93134763:TATGAG>T,SGR003,CTRL051,2\nSGR033,P1349L,chr7:93131926:G>A,chr7:93135203:T>TA,chr7:93134746:AG>A,SGR018,CTRL050,2\nSGR034,R986C,chr7:93133016:G>A,chr7:93132742:C>T,chr7:93132698:A>C,SGR026,CTRL113,0\nSGR034,R986C,chr7:93133016:G>A,chr7:93132742:C>T,chr7:93132814:G>A,SGR025,CTRL114,0\nSGR034,R986C,chr7:93133016:G>A,chr7:93132742:C>T,chr7:93132608:T>A,SGR007,CTRL112,0\nSGR034,R986C,chr7:93133016:G>A,chr7:93132742:C>T,chr7:93132461:C>T,SGR027,CTRL111,0\nSGR034,R986C,chr7:93133016:G>A,chr7:93132742:C>T,chr7:93132052:A>G,SGR046,CTRL110,0\nSGR035,R986C,chr7:93133016:G>A,chr7:93134207:G>A,chr7:93133670:T>A,SGR001,CTRL120,1\nSGR035,R986C,chr7:93133016:G>A,chr7:93134207:G>A,chr7:93133634:G>A,SGR021,CTRL119,1\nSGR035,R986C,chr7:93133016:G>A,chr7:93134207:G>A,chr7:93133587:G>T,SGR052,CTRL118,1\nSGR035,R986C,chr7:93133016:G>A,chr7:93134207:G>A,chr7:93133490:G>A,SGR036,CTRL117,1\nSGR035,R986C,chr7:93133016:G>A,chr7:93134207:G>A,chr7:93134149:C>CTG,SGR028,CTRL121,3\nSGR036,R986H,chr7:93133015:C>T,chr7:93133490:G>A,chr7:93133587:G>T,SGR052,CTRL100,0\nSGR036,R986H,chr7:93133015:C>T,chr7:93133490:G>A,chr7:93133634:G>A,SGR021,CTRL101,0\nSGR036,R986H,chr7:93133015:C>T,chr7:93133490:G>A,chr7:93133670:T>A,SGR001,CTRL102,0\nSGR036,R986H,chr7:93133015:C>T,chr7:93133490:G>A,chr7:93134207:G>A,SGR017,CTRL103,1\nSGR036,R986H,chr7:93133015:C>T,chr7:93133490:G>A,chr7:93133402:AGTGC>A,SGR054,CTRL098,3\nSGR037,R986H,chr7:93133015:C>T,chr7:93132794:C>A,chr7:93132743:G>A,SGR040,CTRL097,0\nSGR037,R986H,chr7:93133015:C>T,chr7:93132794:C>A,chr7:93132432:C>T,SGR031,CTRL093,0\nSGR037,R986H,chr7:93133015:C>T,chr7:93132794:C>A,chr7:93132425:G>A,SGR006,CTRL092,0\nSGR037,R986H,chr7:93133015:C>T,chr7:93132794:C>A,chr7:93132321:G>GA,SGR053,CTRL091,3\nSGR037,R986H,chr7:93133015:C>T,chr7:93132794:C>A,chr7:93132016:TCTTTA>T,SGR005,CTRL089,3\nSGR038,T936I,chr7:93133165:G>A,chr7:93134207:G>A,chr7:93133670:T>A,SGR001,CTRL139,1\nSGR038,T936I,chr7:93133165:G>A,chr7:93134207:G>A,chr7:93133634:G>A,SGR021,CTRL138,1\nSGR038,T936I,chr7:93133165:G>A,chr7:93134207:G>A,chr7:93133587:G>T,SGR052,CTRL137,1\nSGR038,T936I,chr7:93133165:G>A,chr7:93134207:G>A,chr7:93133490:G>A,SGR036,CTRL136,1\nSGR038,T936I,chr7:93133165:G>A,chr7:93134207:G>A,chr7:93134149:C>CTG,SGR028,CTRL140,3\nSGR039,K770E,chr7:93133664:T>C,chr7:93134207:G>A,chr7:93133670:T>A,SGR001,CTRL161,1\nSGR039,K770E,chr7:93133664:T>C,chr7:93134207:G>A,chr7:93134149:C>CTG,SGR028,CTRL162,3\nSGR039,K770E,chr7:93133664:T>C,chr7:93134207:G>A,chr7:93134586:ACTTT>A,SGR029,CTRL163,3\nSGR039,K770E,chr7:93133664:T>C,chr7:93134207:G>A,chr7:93134746:AG>A,SGR018,CTRL164,3\nSGR039,K770E,chr7:93133664:T>C,chr7:93134207:G>A,chr7:93134763:TATGAG>T,SGR003,CTRL165,3\nSGR040,I972T,chr7:93133057:A>G,chr7:93132743:G>A,chr7:93132794:C>A,SGR037,CTRL135,0\nSGR040,I972T,chr7:93133057:A>G,chr7:93132743:G>A,chr7:93132432:C>T,SGR031,CTRL129,0\nSGR040,I972T,chr7:93133057:A>G,chr7:93132743:G>A,chr7:93132425:G>A,SGR006,CTRL128,0\nSGR040,I972T,chr7:93133057:A>G,chr7:93132743:G>A,chr7:93132410:G>A,SGR019,CTRL127,1\nSGR040,I972T,chr7:93133057:A>G,chr7:93132743:G>A,chr7:93132321:G>GA,SGR053,CTRL126,3\nSGR041,I972T,chr7:93133057:A>G,chr7:93132513:TAGG>T,chr7:93132601:CCTAGTG>C,SGR008,CTRL131,1\nSGR041,I972T,chr7:93133057:A>G,chr7:93132513:TAGG>T,chr7:93132461:C>T,SGR027,CTRL130,3\nSGR041,I972T,chr7:93133057:A>G,chr7:93132513:TAGG>T,chr7:93132608:T>A,SGR007,CTRL132,3\nSGR041,I972T,chr7:93133057:A>G,chr7:93132513:TAGG>T,chr7:93132635:C>T,SGR004,CTRL133,3\nSGR041,I972T,chr7:93133057:A>G,chr7:93132513:TAGG>T,chr7:93132689:T>C,SGR045,CTRL134,3\nSGR042,I876T,chr7:93133345:A>G,chr7:93134612:C>T,chr7:93134896:C>T,SGR024,CTRL157,0\nSGR042,I876T,chr7:93133345:A>G,chr7:93134612:C>T,chr7:93134288:C>T,SGR014,CTRL156,0\nSGR042,I876T,chr7:93133345:A>G,chr7:93134612:C>T,chr7:93135134:C>T,SGR023,CTRL158,0\nSGR042,I876T,chr7:93133345:A>G,chr7:93134612:C>T,chr7:93135289:C>T,SGR048,CTRL159,0\nSGR042,I876T,chr7:93133345:A>G,chr7:93134612:C>T,chr7:93135764:G>A,SGR022,CTRL160,0\nSGR043,R986H,chr7:93133015:C>T,chr7:93131401:C>T,chr7:93131783:A>G,SGR009,CTRL088,0\nSGR043,R986H,chr7:93133015:C>T,chr7:93131401:C>T,chr7:93132052:A>G,SGR046,CTRL090,0\nSGR043,R986H,chr7:93133015:C>T,chr7:93131401:C>T,chr7:93132461:C>T,SGR027,CTRL094,0\nSGR043,R986H,chr7:93133015:C>T,chr7:93131401:C>T,chr7:93132608:T>A,SGR007,CTRL095,0\nSGR043,R986H,chr7:93133015:C>T,chr7:93131401:C>T,chr7:93132635:C>T,SGR004,CTRL096,0\nSGR045,R986C,chr7:93133016:G>A,chr7:93132689:T>C,chr7:93132698:A>C,SGR026,CTRL113,0\nSGR045,R986C,chr7:93133016:G>A,chr7:93132689:T>C,chr7:93132608:T>A,SGR007,CTRL112,0\nSGR045,R986C,chr7:93133016:G>A,chr7:93132689:T>C,chr7:93132814:G>A,SGR025,CTRL114,0\nSGR045,R986C,chr7:93133016:G>A,chr7:93132689:T>C,chr7:93132461:C>T,SGR027,CTRL111,0\nSGR045,R986C,chr7:93133016:G>A,chr7:93132689:T>C,chr7:93132052:A>G,SGR046,CTRL110,0\nSGR046,S888F,chr7:93133309:G>A,chr7:93132052:A>G,chr7:93131783:A>G,SGR009,CTRL147,0\nSGR046,S888F,chr7:93133309:G>A,chr7:93132052:A>G,chr7:93132461:C>T,SGR027,CTRL148,0\nSGR046,S888F,chr7:93133309:G>A,chr7:93132052:A>G,chr7:93131401:C>T,SGR043,CTRL146,0\nSGR046,S888F,chr7:93133309:G>A,chr7:93132052:A>G,chr7:93132608:T>A,SGR007,CTRL149,0\nSGR046,S888F,chr7:93133309:G>A,chr7:93132052:A>G,chr7:93132635:C>T,SGR004,CTRL150,0\nSGR047,T1474P,chr7:93131552:T>G,chr7:93134207:G>A,chr7:93132410:G>A,SGR019,CTRL039,0\nSGR047,T1474P,chr7:93131552:T>G,chr7:93134207:G>A,chr7:93133670:T>A,SGR001,CTRL043,1\nSGR047,T1474P,chr7:93131552:T>G,chr7:93134207:G>A,chr7:93133634:G>A,SGR021,CTRL042,1\nSGR047,T1474P,chr7:93131552:T>G,chr7:93134207:G>A,chr7:93133587:G>T,SGR052,CTRL041,1\nSGR047,T1474P,chr7:93131552:T>G,chr7:93134207:G>A,chr7:93133490:G>A,SGR036,CTRL040,1\nSGR048,R986H,chr7:93133015:C>T,chr7:93135289:C>T,chr7:93135134:C>T,SGR023,CTRL107,0\nSGR048,R986H,chr7:93133015:C>T,chr7:93135289:C>T,chr7:93135764:G>A,SGR022,CTRL108,0\nSGR048,R986H,chr7:93133015:C>T,chr7:93135289:C>T,chr7:93134896:C>T,SGR024,CTRL106,0\nSGR048,R986H,chr7:93133015:C>T,chr7:93135289:C>T,chr7:93135916:A>T,SGR013,CTRL109,0\nSGR048,R986H,chr7:93133015:C>T,chr7:93135289:C>T,chr7:93134612:C>T,SGR042,CTRL105,0\nSGR049,R986H,chr7:93133015:C>T,chr7:93133273:T>C,chr7:93133445:G>A,SGR016,CTRL099,0\nSGR049,R986H,chr7:93133015:C>T,chr7:93133273:T>C,chr7:93134288:C>T,SGR014,CTRL104,0\nSGR049,R986H,chr7:93133015:C>T,chr7:93133273:T>C,chr7:93134612:C>T,SGR042,CTRL105,0\nSGR049,R986H,chr7:93133015:C>T,chr7:93133273:T>C,chr7:93134896:C>T,SGR024,CTRL106,0\nSGR049,R986H,chr7:93133015:C>T,chr7:93133273:T>C,chr7:93135134:C>T,SGR023,CTRL107,0\nSGR050,R986H,chr7:93133015:C>T,chr7:93132410:G>A,chr7:93132425:G>A,SGR006,CTRL092,1\nSGR050,R986H,chr7:93133015:C>T,chr7:93132410:G>A,chr7:93132432:C>T,SGR031,CTRL093,1\nSGR050,R986H,chr7:93133015:C>T,chr7:93132410:G>A,chr7:93132743:G>A,SGR040,CTRL097,1\nSGR050,R986H,chr7:93133015:C>T,chr7:93132410:G>A,chr7:93132321:G>GA,SGR053,CTRL091,3\nSGR050,R986H,chr7:93133015:C>T,chr7:93132410:G>A,chr7:93132016:TCTTTA>T,SGR005,CTRL089,3\nSGR051,R1281W,chr7:93132131:T>A,chr7:93131747:G>GT,chr7:93132016:TCTTTA>T,SGR005,CTRL078,0\nSGR051,R1281W,chr7:93132131:T>A,chr7:93131747:G>GT,chr7:93132321:G>GA,SGR053,CTRL079,4\nSGR051,R1281W,chr7:93132131:T>A,chr7:93131747:G>GT,chr7:93133645:T>TC,SGR015,CTRL080,4\nSGR051,R1281W,chr7:93132131:T>A,chr7:93131747:G>GT,chr7:93134149:C>CTG,SGR028,CTRL081,4\nSGR051,R1281W,chr7:93132131:T>A,chr7:93131747:G>GT,chr7:93134586:ACTTT>A,SGR029,CTRL082,4\nSGR052,D1034Y,chr7:93132872:C>A,chr7:93133587:G>T,chr7:93133634:G>A,SGR021,CTRL084,0\nSGR052,D1034Y,chr7:93132872:C>A,chr7:93133587:G>T,chr7:93133670:T>A,SGR001,CTRL086,0\nSGR052,D1034Y,chr7:93132872:C>A,chr7:93133587:G>T,chr7:93133490:G>A,SGR036,CTRL083,0\nSGR052,D1034Y,chr7:93132872:C>A,chr7:93133587:G>T,chr7:93134207:G>A,SGR017,CTRL087,1\nSGR052,D1034Y,chr7:93132872:C>A,chr7:93133587:G>T,chr7:93133645:T>TC,SGR015,CTRL085,3\nSGR053,G1510A,chr7:93131443:C>G,chr7:93132321:G>GA,chr7:93132016:TCTTTA>T,SGR005,CTRL022,0\nSGR053,G1510A,chr7:93131443:C>G,chr7:93132321:G>GA,chr7:93133645:T>TC,SGR015,CTRL023,0\nSGR053,G1510A,chr7:93131443:C>G,chr7:93132321:G>GA,chr7:93131747:G>GT,SGR051,CTRL021,0\nSGR053,G1510A,chr7:93131443:C>G,chr7:93132321:G>GA,chr7:93134149:C>CTG,SGR028,CTRL024,0\nSGR053,G1510A,chr7:93131443:C>G,chr7:93132321:G>GA,chr7:93134586:ACTTT>A,SGR029,CTRL025,0\nSGR054,S1550P,chr7:93131324:A>G,chr7:93133402:AGTGC>A,chr7:93135203:T>TA,SGR033,CTRL005,0\nSGR054,S1550P,chr7:93131324:A>G,chr7:93133402:AGTGC>A,chr7:93133645:T>TC,SGR015,CTRL001,2\nSGR054,S1550P,chr7:93131324:A>G,chr7:93133402:AGTGC>A,chr7:93134149:C>CTG,SGR028,CTRL002,2\nSGR054,S1550P,chr7:93131324:A>G,chr7:93133402:AGTGC>A,chr7:93134586:ACTTT>A,SGR029,CTRL003,2\nSGR054,S1550P,chr7:93131324:A>G,chr7:93133402:AGTGC>A,chr7:93134746:AG>A,SGR018,CTRL004,2\nSGR055,G1510A,chr7:93131443:C>G,chr7:93134986:T>TG,chr7:93134889:TA>T,SGR012,CTRL028,0\nSGR055,G1510A,chr7:93131443:C>G,chr7:93134986:T>TG,chr7:93134763:TATGAG>T,SGR003,CTRL027,0\nSGR055,G1510A,chr7:93131443:C>G,chr7:93134986:T>TG,chr7:93134746:AG>A,SGR018,CTRL026,0\nSGR055,G1510A,chr7:93131443:C>G,chr7:93134986:T>TG,chr7:93134586:ACTTT>A,SGR029,CTRL025,0\nSGR055,G1510A,chr7:93131443:C>G,chr7:93134986:T>TG,chr7:93134149:C>CTG,SGR028,CTRL024,0\nSGR056,V1512A,chr7:93131437:A>G,chr7:93132177:T>A,chr7:93132052:A>G,SGR046,CTRL_SGR056_1,0\nSGR056,V1512A,chr7:93131437:A>G,chr7:93132177:T>A,chr7:93132461:C>T,SGR027,CTRL_SGR056_2,0\nSGR056,V1512A,chr7:93131437:A>G,chr7:93132177:T>A,chr7:93132608:T>A,SGR007,CTRL_SGR056_3,0\nSGR056,V1512A,chr7:93131437:A>G,chr7:93132177:T>A,chr7:93132635:C>T,SGR004,CTRL_SGR056_4,0\nSGR056,V1512A,chr7:93131437:A>G,chr7:93132177:T>A,chr7:93132689:T>C,SGR045,CTRL_SGR056_5,0\nSGR057,K1532N,chr7:93131376:C>A,chr7:93133916:GTTC>G,chr7:93132601:CCTAGTG>C,SGR008,CTRL_SGR057_1,0\nSGR057,K1532N,chr7:93131376:C>A,chr7:93133916:GTTC>G,chr7:93132513:TAGG>T,SGR011,CTRL_SGR057_2,1\nSGR057,K1532N,chr7:93131376:C>A,chr7:93133916:GTTC>G,chr7:93133887:C>G,SGR032,CTRL_SGR057_3,3\nSGR057,K1532N,chr7:93131376:C>A,chr7:93133916:GTTC>G,chr7:93134288:C>T,SGR014,CTRL_SGR057_4,3\nSGR057,K1532N,chr7:93131376:C>A,chr7:93133916:GTTC>G,chr7:93133445:G>A,SGR016,CTRL_SGR057_5,3\n'))
assignments.to_csv(OUT/'control_assignments.csv',index=False)
import gzip, base64, io
expected=pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode('...'))))
actual=pd.read_parquet(PAIR_DIR/'SGR018_metrics.parquet')
keys=['mask_id', 'track_key', 'output_type']
values=['germline_effect', 'somatic_effect', 'combined_cis_effect']
left=expected.set_index(keys)[values].sort_index()
right=actual.set_index(keys)[values].sort_index()
try:
    pd.testing.assert_frame_equal(left,right,check_dtype=False,rtol=1e-6,atol=1e-8)
    status='PASS'
except AssertionError:
    status='DIFFERENT'
(OUT/'comparabilidad.txt').write_text(status+'\n')
print('Comparación con la ejecución archivada:',status)


In [ ]:
from scipy import stats
import shutil
inputs=OUT/'portable_inputs'
inputs.mkdir(exist_ok=True)
RESULTS=OUT/'resultados_finales'
RESULTS.mkdir(exist_ok=True)
def bh(values):
    p=np.asarray(values,float)
    return stats.false_discovery_control(p,method='bh')

import hashlib
ids=set(assignments.observed_pair_id)
assignments.to_csv(inputs/'chromatin_control_assignments_270.csv',index=False)
# The observed distribution defines a single P90 for each scope and modality.
frames=[]
for pid in manifest.Somatic_Event_ID:
    d=pd.read_parquet(PAIR_DIR/f'{pid}_metrics.parquet',columns=['pair_id','background','output_type','track_key','mask_id','is_native_hematopoietic','complete_paired_effects','abs_germline_effect','rescue_to_WT_ratio','closer_to_WT','interaction_log2sum'])
    frames.append(d[(d.mask_id=='PRIMARY_PAIR_UNION') & d.complete_paired_effects & (d.abs_germline_effect>=MIN_ABS_G_EFFECT)])
tracks=pd.concat(frames,ignore_index=True)
tracks.to_parquet(RESULTS/'eligible_chromatin_tracks.parquet',index=False)
parts=[]
threshold_rows=[]
for scope in ['all_hematopoietic','native_hematopoietic']:
    d=tracks if scope=='all_hematopoietic' else tracks[tracks.is_native_hematopoietic]
    thresholds=d[d.pair_id.isin(ids)].groupby('output_type').abs_germline_effect.quantile(.90)
    threshold_rows.extend((scope,k,v) for k,v in thresholds.items())
    d=d[d.abs_germline_effect>=d.output_type.map(thresholds)].copy()
    d['abs_interaction']=d.interaction_log2sum.abs()
    a=d.groupby(['pair_id','background','output_type'],as_index=False).agg(n_tracks=('track_key','nunique'),fraction_closer_to_WT=('closer_to_WT','mean'),median_rescue_to_WT=('rescue_to_WT_ratio','median'),median_interaction_log2sum=('interaction_log2sum','median'),median_abs_interaction_log2sum=('abs_interaction','median'))
    parts.append(a.assign(analysis_scope=scope,mask_id='PRIMARY_PAIR_UNION',variant_subset='all_pairs',track_set='strong_p90'))
d=pd.concat(parts,ignore_index=True)
pd.DataFrame(threshold_rows,columns=['scope','output_type','P90']).to_csv(RESULTS/'observed_thresholds.csv',index=False)
for label,part in [('observed',d[d.pair_id.isin(ids)]),('control',d[~d.pair_id.isin(ids)])]:
    path=inputs/f'{label}_pair_level_chromatin_metrics.csv'
    part.to_csv(path,index=False)
    shutil.copy2(path,RESULTS/path.name)
METRICS=['median_rescue_to_WT', 'fraction_closer_to_WT', 'median_interaction_log2sum', 'median_abs_interaction_log2sum']
def run(events,analysis_dir,out,bh):
    inputs = analysis_dir/'portable_inputs'
    observed = pd.read_csv(inputs/'observed_pair_level_chromatin_metrics.csv')
    controls = pd.read_csv(inputs/'control_pair_level_chromatin_metrics.csv')
    assigned = pd.read_csv(inputs/'chromatin_control_assignments_270.csv')
    retained = set(events.loc[events.sequence,'event'])
    assigned = assigned[assigned.observed_pair_id.isin(retained)]
    assert set(assigned.observed_pair_id)==retained
    assert assigned.groupby('observed_pair_id').size().eq(5).all()
    audit, results, distributions = [], [], {}
    for family, minimum in [('PRIMARY',1),('EFFECTIVE_MIN3',3)]:
        for control_set in ['FULL','STRICT3']:
            mp = assigned
            if control_set == 'STRICT3':
                mp = mp[mp.match_tier<=1]
                mp = mp.groupby('observed_pair_id').filter(lambda x:len(x)>=3)
            for scope in ['native_hematopoietic','all_hematopoietic']:
                for mod in ['DNASE','CHIP_HISTONE','CHIP_TF']:
                    def select(d):
                        return d[(d.analysis_scope==scope)&(d.output_type==mod)&(d.track_set=='strong_p90')&
                                 (d.mask_id=='PRIMARY_PAIR_UNION')&(d.variant_subset=='all_pairs')]
                    obs = select(observed); alt=select(controls).set_index('pair_id'); evs=[]
                    for _,row in obs[obs.pair_id.isin(retained)].sort_values('pair_id').iterrows():
                        ids=mp.loc[mp.observed_pair_id==row.pair_id,'control_pair_id'].tolist()
                        ids=[i for i in ids if i in alt.index and np.isfinite(alt.loc[i,METRICS].to_numpy(float)).all()]
                        vec=row[METRICS].to_numpy(float)
                        if len(ids)>=minimum and np.isfinite(vec).all():
                            assert len(ids)==len(set(ids))
                            evs.append((row.pair_id,row.background,vec,alt.loc[ids,METRICS].to_numpy(float),ids))
                    keys=sorted(set(e[1] for e in evs))
                    stat=np.median(np.stack([np.median(np.stack([e[2] for e in evs if e[1]==k]),axis=0) for k in keys]),axis=0)
                    key='|'.join([family,control_set,scope,mod])
                    rng=np.random.default_rng(np.random.SeedSequence([20260910,int.from_bytes(hashlib.sha256(key.encode()).digest()[:4],'little')]))
                    draw=np.stack([e[3][rng.integers(0,len(e[3]),size=10000)] for e in evs],axis=1)
                    dist=np.median(np.stack([np.median(draw[:,[i for i,e in enumerate(evs) if e[1]==k],:],axis=1) for k in keys],axis=1),axis=1)
                    for e in evs:audit.append([family,control_set,scope,mod,e[0],e[1],len(e[4]),';'.join(e[4])])
                    for k,metric in enumerate(METRICS):
                        x=dist[:,k];center=np.median(x);pid=key+'|'+metric
                        p=(1+np.count_nonzero(np.abs(x-center)>=abs(stat[k]-center)))/10001
                        distributions[pid]=x
                        results.append([pid,family,control_set,scope,mod,metric,'PRIMARY_PAIR_UNION','strong_p90','all_pairs',len(evs),len(keys),minimum,stat[k],center,np.quantile(x,.025),np.quantile(x,.975),stat[k]-center,p,48])
    columns=['test_id','analysis_family','control_set','analysis_scope','output_type','metric','mask_id','track_set','variant_subset','n_events','n_backgrounds','min_effective_controls','observed_stat','null_median','null_q025','null_q975','effect_vs_null_median','p_empirical_two_sided','family_n_tests']
    d=pd.DataFrame(results,columns=columns); d['q_family_global']=d.groupby('analysis_family').p_empirical_two_sided.transform(bh)
    d.to_csv(out/'chromatin_tests.csv',index=False)
    pd.DataFrame(audit,columns=['family','control_set','scope','modality','event','background','used_controls','control_ids']).to_csv(out/'chromatin_control_audit.csv',index=False)
    np.savez_compressed(out/'chromatin_null.npz',**distributions)
    return {'assignments':len(assigned),'unique_controls':assigned.control_pair_id.nunique()}

events=pd.DataFrame({'event':sorted(ids),'sequence':True})
run(events,OUT,RESULTS,bh)
shutil.copy2(OUT/'analysis_config.json',RESULTS/'analysis_config.json')
manifest.to_csv(RESULTS/'configuration_manifest.csv',index=False)

from google.colab import files
for path in [OUT/'comparabilidad.txt', OUT/'run_audit.csv']:
    shutil.copy2(path, RESULTS/path.name)
assignments.to_csv(RESULTS/'control_assignments.csv',index=False)
files.download(shutil.make_archive('/content/SAMD9L_FINAL54_cromatina','zip',RESULTS))
